In [1]:
# ======================================================
# Notebook: PINN_2.3_SHM_Z24 (V1)
# Auteur : Abdellah Riyahi
# Objectif : Implémentation et entraînement d'un Physics-Informed Neural Network (PINN)
#            pour la détection de dommages structurels sur le pont Z24 (PDT).
# Version : 2.3 (évoluée depuis 2.2 avec corrections & optimisations)
# Date : 2025-08-19
# ======================================================

import os
import sys
import math
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, losses

from pathlib import Path
from datetime import datetime

print("[INFO] TensorFlow:", tf.__version__)

# Fixer la seed pour reproductibilité
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Racine projet (adapter si nécessaire au HPC ou local)
PROJECT_ROOT = Path("/home/a.riyahi/spinn_project/")
DATASET_ROOT = PROJECT_ROOT / "DatasetPDT"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
MODELS_DIR   = PROJECT_ROOT / "models"

# Création des répertoires si inexistants
for d in [OUTPUTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("[INFO] Racine projet :", PROJECT_ROOT)
print("[INFO] Dossier dataset :", DATASET_ROOT)
print("[INFO] Dossier outputs :", OUTPUTS_DIR)
print("[INFO] Dossier modèles :", MODELS_DIR)


2025-11-10 19:26:13.958821: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-10 19:26:14.087219: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-10 19:26:14.103463: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-10 19:26:15.900855: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


[INFO] TensorFlow: 2.12.0
[INFO] Racine projet : /home/a.riyahi/spinn_project
[INFO] Dossier dataset : /home/a.riyahi/spinn_project/DatasetPDT
[INFO] Dossier outputs : /home/a.riyahi/spinn_project/outputs
[INFO] Dossier modèles : /home/a.riyahi/spinn_project/models


In [2]:
# ======================================================
# Cellule 0.1 : Configuration physique & infos runtime
# (robuste, auto‑complète les clés manquantes et enregistre)
# ======================================================

import os, json, math, platform
from pathlib import Path
from datetime import datetime
import tensorflow as tf
import numpy as np

VERSION = "PINN_2.3"

# Répertoires dérivés (sous PROJECT_ROOT définie en Cellule 0)
ARTIFACTS_DIR = OUTPUTS_DIR / "artifacts"
FIGS_DIR      = OUTPUTS_DIR / "figs"
CACHE_DIR     = OUTPUTS_DIR / "cache"
for d in [ARTIFACTS_DIR, FIGS_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def _save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

# --- Détection GPU / runtime ---
physical_gpus = tf.config.list_physical_devices("GPU")
logical_gpus  = tf.config.list_logical_devices("GPU")
print(f"[RUNTIME] Python : {platform.python_version()}")
print(f"[RUNTIME] TF     : {tf.__version__}")
print(f"[RUNTIME] CUDA/GPU: {len(physical_gpus)} physical / {len(logical_gpus)} logical")
for i, g in enumerate(physical_gpus):
    print(f"          └─ GPU[{i}]:", g)

# Politique de précision (float32 ⇒ stable sur CPU/GPU)
tf.keras.mixed_precision.set_global_policy("float32")

# --- PHYS_CFG : valeurs par défaut + complétion sûre ---
#   - zeta : amortissement (adimensionnel)
#   - omega0_rad_s : pulsation propre rad/s (≈ 2π·f0). On garde 4 Hz par défaut.
#   - fs_hz : fréquence d’échantillonnage (utile pour PSD)
#   - L : longueur des signaux (doit matcher le dataset, 65536 par défaut)
#   - time_scale_s : durée physique de la trace. None ⇒ pas de rescale; si connu ⇒ (L-1)/fs_hz
PHYS_CFG = {
    "version": VERSION,
    "zeta": 0.01,
    "omega0_rad_s": float(2.0 * math.pi * 4.0),  # 4 Hz ⇒ ~25.1327 rad/s
    "fs_hz": 100.0,
    "L": 65536,
    "time_scale_s": None,            # restera None si inconnu (pas de rescale)
    # hypers par défaut (overridables plus tard)
    "seed": 42,
    "n_coll": 2048,
    "mlp_widths": [64, 64, 64],
    "activation": "tanh",
    "lr_phys": 1e-3,
    "epochs_phys": 2000,
    "patience_phys": 200,
    # Cellule 4 (hybride)
    "lr": 1e-3,
    "min_lr": 1e-5,
    "epochs_cell4": 1500,
    "patience_cell4": 200,
    "w_phys": 1.0,
    "w_data": 0.0,
    "batch_data": 1024,
}

# Si tu as déjà un fichier 2.2, on peut l’importer et mettre à jour proprement
prev_cfg_path = ARTIFACTS_DIR / "pinn_phys_config_v2_2.json"
if prev_cfg_path.exists():
    try:
        with open(prev_cfg_path, "r") as f:
            prev = json.load(f)
        # override soft (on ne garde pas d’anciennes clés douteuses)
        for k in ["zeta","omega0_rad_s","fs_hz","L","time_scale_s",
                  "seed","n_coll","mlp_widths","activation",
                  "lr_phys","epochs_phys","patience_phys",
                  "lr","min_lr","epochs_cell4","patience_cell4",
                  "w_phys","w_data","batch_data"]:
            if k in prev and prev[k] is not None:
                PHYS_CFG[k] = prev[k]
        print("[INFO] PHYS_CFG importé depuis v2.2 et mis à jour pour v2.3.")
    except Exception as e:
        print(f"[WARN] Impossible de lire l’ancienne config v2.2: {e}")

# Si time_scale_s est None mais fs_hz & L connus, on propose une valeur physique
if PHYS_CFG.get("time_scale_s", None) is None and PHYS_CFG.get("fs_hz", 0) and PHYS_CFG.get("L", 0):
    PHYS_CFG["time_scale_s"] = (float(PHYS_CFG["L"]) - 1.0) / float(PHYS_CFG["fs_hz"])

# Conversions/typage sûr
PHYS_CFG["zeta"]          = float(PHYS_CFG["zeta"])
PHYS_CFG["omega0_rad_s"]  = float(PHYS_CFG["omega0_rad_s"])
PHYS_CFG["fs_hz"]         = float(PHYS_CFG["fs_hz"])
PHYS_CFG["L"]             = int(PHYS_CFG["L"])
PHYS_CFG["time_scale_s"]  = float(PHYS_CFG["time_scale_s"]) if PHYS_CFG["time_scale_s"] is not None else 0.0

# Sauvegarde config v2.3
cfg_path = ARTIFACTS_DIR / "pinn_phys_config_v2_3.json"
_save_json(PHYS_CFG, cfg_path)

# Journal de run
run_info = {
    "notebook": "PINN_2.3_SHM_Z24.ipynb",
    "version": VERSION,
    "when": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "device_count": {
        "gpu_physical": len(physical_gpus),
        "gpu_logical": len(logical_gpus)
    },
    "paths": {
        "project_root": str(PROJECT_ROOT),
        "dataset_root": str(DATASET_ROOT),
        "outputs_dir":  str(OUTPUTS_DIR),
        "models_dir":   str(MODELS_DIR),
        "artifacts_dir":str(ARTIFACTS_DIR)
    },
    "seed": int(PHYS_CFG["seed"])
}
_save_json(run_info, ARTIFACTS_DIR / "run_info_v2_3.json")

print(f"[SAVE] PHYS_CFG -> {cfg_path}")
print(f"[SAVE] run_info -> {ARTIFACTS_DIR / 'run_info_v2_3.json'}")
print(f"[PHYS] zeta={PHYS_CFG['zeta']:.4f}, omega0={PHYS_CFG['omega0_rad_s']:.4f} rad/s, "
      f"fs={PHYS_CFG['fs_hz']:.1f} Hz, L={PHYS_CFG['L']}, time_scale_s={PHYS_CFG['time_scale_s']:.6f}")


[RUNTIME] Python : 3.11.7
[RUNTIME] TF     : 2.12.0
[RUNTIME] CUDA/GPU: 0 physical / 0 logical
[INFO] PHYS_CFG importé depuis v2.2 et mis à jour pour v2.3.
[SAVE] PHYS_CFG -> /home/a.riyahi/spinn_project/outputs/artifacts/pinn_phys_config_v2_3.json
[SAVE] run_info -> /home/a.riyahi/spinn_project/outputs/artifacts/run_info_v2_3.json
[PHYS] zeta=0.0100, omega0=25.1327 rad/s, fs=100.0 Hz, L=65536, time_scale_s=655.350000


In [3]:
# ============================================================
# Cellule 1 — Inventaire Z24 & Manifest (V2.3 robuste)
#  - Scanne DatasetPDT pour classes {01,03,04,05,06}
#  - Support .mat v5 (scipy) et v7.3/HDF5 (h5py)
#  - Heuristique de variable la plus longue
#  - Ecrit artifacts/manifest.json + résumé console
# ============================================================

from __future__ import annotations
import os, json, re, math, random, datetime as dt
from pathlib import Path
from typing import List, Dict, Any

import numpy as np

# --- 1.0 Contexte chemins / seed (hérite de la Cellule 0 si dispo) ---
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path("/home/a.riyahi/spinn_project")
if 'DATASET_ROOT' not in globals():
    DATASET_ROOT = PROJECT_ROOT / "DatasetPDT"
if 'OUTPUTS_ROOT' not in globals():
    OUTPUTS_ROOT = PROJECT_ROOT / "outputs_pinn_v2_3"
if 'ARTIFACTS_DIR' not in globals():
    ARTIFACTS_DIR = OUTPUTS_ROOT / "artifacts"

for d in [OUTPUTS_ROOT, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = int(globals().get('SEED', 42))
random.seed(SEED)
np.random.seed(SEED)

CLASSES: List[str] = ['01', '03', '04', '05', '06']   # classes retenues Z24

# --- 1.1 Chargeurs .mat robustes ---
try:
    import scipy.io as sio  # pour v5
except Exception:
    sio = None

try:
    import h5py  # pour v7.3 (HDF5)
except Exception:
    h5py = None

def _is_hdf5(path: Path) -> bool:
    """Détecte un .mat v7.3/HDF5 (si h5py dispo)."""
    if h5py is None:
        return False
    try:
        with h5py.File(path, "r"):
            return True
    except Exception:
        return False

def _peek_mat_vars_v5(path: Path) -> dict:
    """Lecture légère des variables (v5) sans charger les gros arrays."""
    if sio is None:
        return {"format": "unknown", "vars": []}
    try:
        md = sio.whosmat(str(path))  # [(name, shape, isglobal), ...]
        vars_meta = [{"name": n, "shape": tuple(shp)} for (n, shp, _) in md]
        return {"format": "v5", "vars": vars_meta}
    except Exception:
        return {"format": "unknown", "vars": []}

def _peek_mat_vars_h5(path: Path) -> dict:
    """Liste les datasets HDF5 (v7.3)."""
    if h5py is None:
        return {"format": "unknown", "vars": []}
    try:
        vars_meta = []
        with h5py.File(path, "r") as f:
            def visit_fn(name, obj):
                if isinstance(obj, h5py.Dataset):
                    vars_meta.append({"name": name, "shape": tuple(obj.shape)})
            f.visititems(visit_fn)
        return {"format": "v7.3", "vars": vars_meta}
    except Exception:
        return {"format": "unknown", "vars": []}

def _peek_mat(path: Path) -> dict:
    """Retourne format + liste (name,shape) + candidate_longest_var."""
    if _is_hdf5(path):
        meta = _peek_mat_vars_h5(path)
    else:
        meta = _peek_mat_vars_v5(path)

    candidate = None
    best_len = -1
    for v in meta.get("vars", []):
        shp = v.get("shape", ())
        if not shp:
            continue
        n = int(np.prod(shp))
        if n > best_len:
            best_len = n
            candidate = v.get("name")
    meta["candidate_longest_var"] = candidate
    return meta

def _natural_key(s: str):
    """Tri naturel : e.g., setup2 < setup10."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

def _list_mat_files(root: Path, classes: List[str]) -> Dict[str, List[Path]]:
    """Retourne un dict {classe: [paths]}.
       Recherche prioritairement root/<cls>/avt/*.mat puis fallback root/**/*.mat."""
    per_class: Dict[str, List[Path]] = {c: [] for c in classes}
    for c in classes:
        # préférence pour la hiérarchie standard <cls>/avt
        pri = sorted((root / c / "avt").glob("*.mat"), key=lambda p: _natural_key(p.name))
        if pri:
            per_class[c] = pri
            continue
        # fallback : scan récursif si pas de sous-dossier "avt"
        rec = sorted((root / c).rglob("*.mat"), key=lambda p: _natural_key(p.name))
        per_class[c] = [p for p in rec if p.is_file()]
    return per_class

def _sample_peeks(files: List[Path], k=3) -> List[dict]:
    """Sonde jusqu'à k fichiers par classe pour le manifest."""
    if len(files) == 0:
        return []
    picks = random.sample(files, k=min(k, len(files)))
    out = []
    for p in picks:
        meta = _peek_mat(p)
        out.append({
            "file": p.name,
            "rel_dir": str(p.parent.relative_to(DATASET_ROOT)) if DATASET_ROOT in p.parents else p.parent.name,
            "fmt": meta.get("format", "unknown"),
            "vars": len(meta.get("vars", [])),
            "candidate_longest_var": meta.get("candidate_longest_var", None)
        })
    return out

# --- 1.2 Scan principal ---
print(f"[SCAN] Racine dataset : {DATASET_ROOT}")
per_class_files = _list_mat_files(DATASET_ROOT, CLASSES)
counts = {c: len(per_class_files[c]) for c in CLASSES}
total_mat = int(sum(counts.values()))

missing = [c for c in CLASSES if counts[c] == 0]
if missing:
    print(f"[WARN] Aucune donnée trouvée pour classes: {missing} (sous {DATASET_ROOT})")

# --- 1.3 Sondage & manifest ---
samples = {c: _sample_peeks(per_class_files[c], k=3) for c in CLASSES}

manifest = {
    "version": "PINN_2.3",
    "built_at": dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "root": str(DATASET_ROOT),
    "classes": CLASSES,
    "counts": counts,
    "total_mat": total_mat,
    "samples": samples,
    "notes": {
        "peek_support": {
            "scipy_v5": bool(sio is not None),
            "h5py_v7_3": bool(h5py is not None)
        },
        "seed": SEED
    }
}

man_path = ARTIFACTS_DIR / "manifest.json"
with open(man_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print(f"[SAVE] manifest -> {man_path}")

# --- 1.4 Résumé console lisible ---
print("\n=== MANIFEST RÉSUMÉ ===")
print(f"Racine : {DATASET_ROOT}")
print(f"Version: {manifest['version']}")
print(f"Construit le: {manifest['built_at']}")
print(f"Total .mat: {manifest['total_mat']}")
print("Par classe:")
for c in CLASSES:
    print(f"  - {c}: {counts[c]}")
print("\nÉchantillons sondés (max 3 / classe) :")
for c in CLASSES:
    if not samples[c]:
        continue
    for s in samples[c]:
        print(f"  [{c}] {s['file']} | dir={s['rel_dir']} | fmt={s['fmt']} "
              f"| vars={s['vars']} | longest={s['candidate_longest_var']}")
print("Cellule 1 terminée ✅ — inventaire prêt.")


[SCAN] Racine dataset : /home/a.riyahi/spinn_project/DatasetPDT
[SAVE] manifest -> /home/a.riyahi/spinn_project/outputs/artifacts/manifest.json

=== MANIFEST RÉSUMÉ ===
Racine : /home/a.riyahi/spinn_project/DatasetPDT
Version: PINN_2.3
Construit le: 2025-11-10 19:26:22
Total .mat: 45
Par classe:
  - 01: 9
  - 03: 9
  - 04: 9
  - 05: 9
  - 06: 9

Échantillons sondés (max 3 / classe) :
  [01] 01setup02.mat | dir=01/avt | fmt=v5 | vars=2 | longest=data
  [01] 01setup01.mat | dir=01/avt | fmt=v5 | vars=2 | longest=data
  [01] 01setup06.mat | dir=01/avt | fmt=v5 | vars=2 | longest=data
  [03] 03setup05.mat | dir=03/avt | fmt=v5 | vars=2 | longest=data
  [03] 03setup04.mat | dir=03/avt | fmt=v5 | vars=2 | longest=data
  [03] 03setup02.mat | dir=03/avt | fmt=v5 | vars=2 | longest=data
  [04] 04setup03.mat | dir=04/avt | fmt=v5 | vars=2 | longest=data
  [04] 04setup02.mat | dir=04/avt | fmt=v5 | vars=2 | longest=data
  [04] 04setup06.mat | dir=04/avt | fmt=v5 | vars=2 | longest=data
  [05] 05s

In [4]:
# === Cellule 2 — Chargement Z24 + Manifeste + Splits (PINN v2.3 corrigée) ===
# Entrées: DatasetPDT/<cls>/avt/*.mat  (cls in {'01','03','04','05','06'})
# Sorties:  - outputs/cache/pinn2_2_dataset_cache.npz
#           - outputs/artifacts/manifest_pinn2_2.json
#           - outputs/artifacts/pinn_phys_config_v2_2.json
# Hyp: scipy, scikit-learn installés.

from __future__ import annotations
import os, re, json, time, math
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------- 2.0 Contexte & chemins ----------
PROJ = Path("/home/a.riyahi/spinn_project")
DATA_ROOT = PROJ / "DatasetPDT"
OUT_DIR   = PROJ / "outputs"
ART_DIR   = OUT_DIR / "artifacts"
CACHE_DIR = OUT_DIR / "cache"
for d in (OUT_DIR, ART_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- 2.1 Configuration centrale ----------
CFG = {
    "version": "PINN_2.3",
    "classes": ["01", "03", "04", "05", "06"],
    "signal_len": 65536,            # L
    "norm": "zscore",               # normalisation par StandardScaler (fit sur TRAIN)
    "rng_seed": 42,
    "data_root": str(DATA_ROOT),
    "built_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

np.random.seed(CFG["rng_seed"])

# ---------- 2.2 Utilitaires ----------
def _natural_key(s: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

def list_mat_files_for_class(c: str) -> List[Path]:
    """Fichiers attendus dans DATA_ROOT/<cls>/avt/*.mat"""
    ptn = DATA_ROOT / c / "avt"
    return sorted(ptn.glob("*.mat"), key=lambda p: _natural_key(p.name))

def _pick_first_ndarray(d: dict) -> np.ndarray|None:
    for k, v in d.items():
        if isinstance(v, np.ndarray) and v.dtype != np.object_:
            return v
    return None

def _choose_channels_as_first_axis(arr: np.ndarray) -> np.ndarray:
    """Retourne (n_channels, n_pts). Heuristique: si (n_pts, n_channels) -> transpose."""
    if arr.ndim == 1:
        return arr[None, :]
    if arr.shape[0] < arr.shape[1]:
        return arr.T
    return arr

def load_one_mat_as_1D(mat_path: Path, target_len: int) -> np.ndarray:
    """Charge un .mat, récupère la variable plausible, choisit le canal le plus long,
    pad/trim à target_len. Renvoie 1D float32 (n_pts,)."""
    mat = loadmat(mat_path)
    var = None
    for k in ("data", "Data", "DATA"):
        if k in mat:
            var = mat[k]
            break
    if var is None:
        var = _pick_first_ndarray(mat)
    if var is None:
        raise RuntimeError(f"Aucune variable ndarray exploitable dans {mat_path.name}")

    arr = np.array(var, dtype=np.float32)
    chans = _choose_channels_as_first_axis(arr)  # (n_ch, n_pts)

    # canal le plus long (sécurité, souvent identiques)
    idx = int(np.argmax([ch.shape[-1] for ch in chans]))
    sig = np.asarray(chans[idx], dtype=np.float32)

    # pad/trim à target_len (padding dernière valeur, robustesse scripts Z24)
    if sig.size < target_len:
        pad_val = float(sig[-1]) if sig.size > 0 else 0.0
        pad = np.full((target_len - sig.size,), pad_val, dtype=np.float32)
        sig = np.concatenate([sig, pad], axis=0)
    elif sig.size > target_len:
        sig = sig[:target_len]

    # guard NaN/inf
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return sig

def build_manifest(files_by_cls: Dict[str, List[Path]]) -> Dict[str, Any]:
    counts = {c: len(fs) for c, fs in files_by_cls.items()}
    samples = []
    for c, files in files_by_cls.items():
        for p in files[:3]:
            samples.append(f"[{c}] {p.name} | fmt=v5-ish | candidate_longest_var=data")
    return {
        "root": str(DATA_ROOT),
        "version": CFG["version"],
        "built_at": CFG["built_at"],
        "total_mat": sum(counts.values()),
        "per_class": counts,
        "samples": samples,
    }

# ---------- 2.3 Indexation des fichiers ----------
classes = CFG["classes"]
files_by_cls: Dict[str, List[Path]] = {c: list_mat_files_for_class(c) for c in classes}
missing = [c for c, fs in files_by_cls.items() if len(fs) == 0]
if missing:
    raise FileNotFoundError(
        f"Aucun .mat trouvé pour classes {missing} sous {DATA_ROOT}/<cls>/avt/*.mat"
    )

# ---------- 2.4 Lecture & empilement ----------
X_list: List[np.ndarray] = []
y_list: List[int] = []
L = int(CFG["signal_len"])

for c in classes:
    label = classes.index(c)
    for mat_path in files_by_cls[c]:
        sig = load_one_mat_as_1D(mat_path, L)   # (L,)
        X_list.append(sig)
        y_list.append(label)

X = np.stack(X_list, axis=0).astype(np.float32)   # (N, L)
y = np.asarray(y_list, dtype=np.int64)            # (N,)
assert X.shape[0] == y.shape[0], "X et y doivent avoir même N"

# ---------- 2.5 Splits stratifiés 70/15/15 ----------
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=CFG["rng_seed"], stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=CFG["rng_seed"], stratify=y_tmp
)
del X_tmp, y_tmp

# ---------- 2.6 Normalisation ----------
if CFG["norm"] == "zscore":
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)
else:
    scaler = None

# ---------- 2.7 Mise en forme Keras/PINN: (N, L, 1) ----------
X_train = X_train[..., None]
X_val   = X_val[..., None]
X_test  = X_test[..., None]

# ---------- 2.8 Sauvegardes: cache + manifeste ----------
cache_path = CACHE_DIR / "pinn2_2_dataset_cache.npz"
np.savez_compressed(
    cache_path,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
    classes=np.array(classes)
)

manifest = build_manifest(files_by_cls)
manifest_path = ART_DIR / "manifest_pinn2_2.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# ---------- 2.9 Config physique exportée (pour Cellules 3–6) ----------
# Si tu connais l'échantillonnage réel, mets-le dans l'ENV: SPINN_FS_HZ
# Sinon on garde un défaut cohérent (100 Hz).
fs_hz = float(os.environ.get("SPINN_FS_HZ", "100.0"))
time_scale_s = (L - 1) / fs_hz if fs_hz > 0 else 0.0

phys_cfg = {
    # paramètres SDOF de base (ajuste au besoin)
    "zeta": 0.01,
    "omega0_rad_s": float(2.0 * math.pi * 4.0),  # 4 Hz ≈ 25.1327 rad/s
    # échelle temporelle (clé du rescaling des dérivées)
    "fs_hz": fs_hz,
    "L": int(L),
    "time_scale_s": float(time_scale_s),         # = (L-1)/fs_hz
    # hyperparams par défaut (surchargés plus tard si besoin)
    "seed": CFG["rng_seed"],
    "n_coll": 2048,
    "mlp_widths": [64, 64, 64],
    "activation": "tanh",
    "lr_phys": 1e-3,
    "epochs_phys": 2000,
    "patience_phys": 200,
    # Cellule 4
    "lr": 1e-3,
    "min_lr": 1e-5,
    "epochs_cell4": 1500,
    "patience_cell4": 200,
    "w_phys": 1.0,
    "w_data": 0.0,
    "batch_data": 1024
}
phys_cfg_path = ART_DIR / "pinn_phys_config_v2_2.json"
with open(phys_cfg_path, "w") as f:
    json.dump(phys_cfg, f, indent=2)

# ---------- 2.10 Journalisation ----------
print("=== MANIFEST RÉSUMÉ ===")
print(f"Racine : {DATA_ROOT}")
print(f"Version: {CFG['version']}")
print(f"Construit le: {CFG['built_at']}")
tot = sum(len(v) for v in files_by_cls.values())
print(f"Total .mat: {tot}")
print("Par classe:")
for c in classes:
    print(f"  - {c}: {len(files_by_cls[c])}")

print("\n[SHAPES]")
print(f"Xtr={X_train.shape}, Xva={X_val.shape}, Xte={X_test.shape} | "
      f"ytr={y_train.shape}, yva={y_val.shape}, yte={y_test.shape}")

print(f"[SAVE] cache      -> {cache_path}")
print(f"[SAVE] manifeste  -> {manifest_path}")
print(f"[SAVE] phys cfg   -> {phys_cfg_path}")
print(f"[INFO] fs_hz={fs_hz:.3f} | L={L} | time_scale_s={time_scale_s:.6f}s")
print("Cellule 2 terminée ✅")


=== MANIFEST RÉSUMÉ ===
Racine : /home/a.riyahi/spinn_project/DatasetPDT
Version: PINN_2.3
Construit le: 2025-11-10 19:26:23
Total .mat: 45
Par classe:
  - 01: 9
  - 03: 9
  - 04: 9
  - 05: 9
  - 06: 9

[SHAPES]
Xtr=(31, 65536, 1), Xva=(7, 65536, 1), Xte=(7, 65536, 1) | ytr=(31,), yva=(7,), yte=(7,)
[SAVE] cache      -> /home/a.riyahi/spinn_project/outputs/cache/pinn2_2_dataset_cache.npz
[SAVE] manifeste  -> /home/a.riyahi/spinn_project/outputs/artifacts/manifest_pinn2_2.json
[SAVE] phys cfg   -> /home/a.riyahi/spinn_project/outputs/artifacts/pinn_phys_config_v2_2.json
[INFO] fs_hz=100.000 | L=65536 | time_scale_s=655.350000s
Cellule 2 terminée ✅


In [5]:
# ============================
# Cellule 3 — PINN (SDOF) : modèle + entraînement sur L_phys (corrigée V2.2→V2.3)
# Objectif : entraîner u(t_norm) avec la contrainte physique seule (pas de terme data ici)
# Points clés :
#  - time_scale_s sécurisé (jamais None) + rescaling correct de u̇ et ü
#  - L lu depuis le cache (fallback 65536 si absent)
#  - échantillonnage de collocation stable (indices uniques, top-up si besoin)
#  - early stopping + best checkpoint
# ============================

from __future__ import annotations
import os, json, time, math
from pathlib import Path
import numpy as np
import tensorflow as tf

# --- 3.0  Chemins & artefacts ---
PROJ_DIR = Path("/home/a.riyahi/spinn_project")
OUT_DIR  = PROJ_DIR / "outputs"
ART_DIR  = OUT_DIR / "artifacts"
CACHE_DIR= OUT_DIR / "cache"
MODEL_DIR= PROJ_DIR / "models"
for d in (OUT_DIR, ART_DIR, CACHE_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- 3.1  Longueur L (depuis le cache si dispo) ---
DEFAULT_L = 65536
try:
    ds = np.load(CACHE_DIR / "pinn2_2_dataset_cache.npz", allow_pickle=False)
    L = int(ds["X_train"].shape[1])
except Exception:
    L = DEFAULT_L

# --- 3.2  Configuration physique (robuste) ---
phys_cfg_path = ART_DIR / "pinn_phys_config_v2_2.json"
if phys_cfg_path.exists():
    with open(phys_cfg_path, "r") as f:
        PHYS_CFG = json.load(f)
else:
    PHYS_CFG = {}

# sécurisation + valeurs par défaut
zeta         = float(PHYS_CFG.get("zeta", 0.01))
omega0_rad_s = float(PHYS_CFG.get("omega0_rad_s", 2.0*math.pi*4.0))  # 4 Hz -> 25.1327 rad/s
fs_hz        = float(PHYS_CFG.get("fs_hz", 100.0))
# IMPORTANT : time_scale_s peut être None dans d’anciennes cellules -> on force un float valide
_time_scale  = PHYS_CFG.get("time_scale_s", (L-1)/fs_hz if fs_hz > 0 else 0.0)
try:
    time_scale_s = float(_time_scale) if _time_scale is not None else 0.0
except Exception:
    time_scale_s = 0.0

# On logge (utile pour tracer l’origination des valeurs)
print("[PHYS] zeta=%.4f, omega0=%.4f rad/s, time_scale_s=%s" %
      (zeta, omega0_rad_s, f"{time_scale_s:.6f}" if time_scale_s>0 else "None"))

# --- 3.3  Hyperparamètres (écrasables par PHYS_CFG) ---
HP = {
    "seed":        42,
    "n_coll":      2048,
    "widths":      [64, 64, 64],
    "activation":  "tanh",
    "lr":          1e-3,
    "epochs":      1500,
    "patience":    200,
}
HP["seed"]        = int(PHYS_CFG.get("seed", HP["seed"]))
HP["n_coll"]      = int(PHYS_CFG.get("n_coll", HP["n_coll"]))
HP["activation"]  = str(PHYS_CFG.get("activation", HP["activation"]))
HP["lr"]          = float(PHYS_CFG.get("lr_phys", HP["lr"]))
HP["epochs"]      = int(PHYS_CFG.get("epochs_phys", HP["epochs"]))
HP["patience"]    = int(PHYS_CFG.get("patience_phys", HP["patience"]))
if "mlp_widths" in PHYS_CFG:
    HP["widths"] = list(PHYS_CFG["mlp_widths"])

tf.keras.utils.set_random_seed(HP["seed"])
print("[HP]   n_coll=%d, widths=%s, act=%s, lr=%.1e, epochs=%d, patience=%d" %
      (HP["n_coll"], HP["widths"], HP["activation"], HP["lr"], HP["epochs"], HP["patience"]))

# --- 3.4  Grille temporelle normalisée ---
def make_time_grid(L: int) -> tf.Tensor:
    return tf.reshape(
        tf.linspace(tf.constant(0.0, tf.float32), tf.constant(1.0, tf.float32), L),
        (-1, 1)
    )

t_norm_all = make_time_grid(L)

# --- 3.5  Échantillonnage collocation (indices uniques + top-up) ---
def sample_collocation(L: int, n: int, seed: int) -> tf.Tensor:
    rng = np.random.default_rng(seed)
    idx = rng.integers(low=0, high=L, size=n, endpoint=False)
    idx = np.unique(idx)
    while idx.size < n:
        extra = rng.integers(low=0, high=L, size=(n-idx.size), endpoint=False)
        idx = np.unique(np.concatenate([idx, extra]))
    idx_tf = tf.convert_to_tensor(idx.astype(np.int32))
    return tf.gather(t_norm_all, idx_tf)

# --- 3.6  Physique : résidu SDOF + dérivées (avec rescale temps) ---
@tf.function
def h_phys_sdof(u, u_dot, u_2dot, zeta, omega0):
    z = tf.cast(zeta, tf.float32)
    w = tf.cast(omega0, tf.float32)
    return u_2dot + 2.0*z*w*u_dot + (w*w)*u

@tf.function
def derivatives_from_model(model, t, time_scale_s: float):
    t = tf.cast(t, tf.float32)  # (B,1), t_norm ∈ [0,1]
    with tf.GradientTape(persistent=True) as g2:
        g2.watch(t)
        with tf.GradientTape() as g1:
            g1.watch(t)
            u = model(t, training=True)    # (B,1)
        u_dot = g1.gradient(u, t)          # du/dt_norm
    u_2dot = g2.gradient(u_dot, t)         # d2u/dt_norm2
    del g1, g2

    # Rescale vers le temps physique si time_scale_s>0 (dt = dt_norm * time_scale_s)
    if time_scale_s and time_scale_s > 0.0:
        invT = tf.constant(1.0/time_scale_s, tf.float32)
        u_dot  = u_dot  * invT
        u_2dot = u_2dot * (invT*invT)
    return u, u_dot, u_2dot

@tf.function
def L_phys(model, t_subset, zeta, omega0, time_scale_s: float):
    u, u_dot, u_2dot = derivatives_from_model(model, t_subset, time_scale_s)
    r = h_phys_sdof(u, u_dot, u_2dot, zeta, omega0)
    return tf.reduce_mean(tf.square(r))

# --- 3.7  Modèle PINN ---
def build_pinn(widths=(64,64,64), act="tanh") -> tf.keras.Model:
    Lyrs = tf.keras.layers
    inp = Lyrs.Input(shape=(1,), name="t_norm")
    x = inp
    for i, w in enumerate(widths, 1):
        x = Lyrs.Dense(w, activation=act, name=f"mlp_{i}")(x)
    out = Lyrs.Dense(1, activation=None, name="u")(x)  # sortie scalaire (None,1)
    return tf.keras.Model(inp, out, name="PINN_u_of_t")

pinn = build_pinn(widths=HP["widths"], act=HP["activation"])
pinn.summary()

# --- 3.8  Entraînement (physique uniquement) ---
opt = tf.keras.optimizers.Adam(learning_rate=HP["lr"])
EPOCHS   = HP["epochs"]
PATIENCE = HP["patience"]

best_loss = np.inf
pat       = 0
stamp     = time.strftime("%Y%m%d_%H%M%S")
best_path = MODEL_DIR / f"pinn_u_only_best_{stamp}.h5"

@tf.function
def train_step(model, t_subset, zeta, omega0, time_scale_s):
    with tf.GradientTape() as tape:
        loss = L_phys(model, t_subset, zeta, omega0, time_scale_s)
    grads = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    return loss

print(f"[PINN] Entraînement L_phys | L={L} | n_coll/epoch={HP['n_coll']} | epochs={EPOCHS} | lr={HP['lr']:.1e}")
for ep in range(1, EPOCHS+1):
    # nouvel échantillon à chaque epoch (meilleure couverture)
    t_subset = sample_collocation(L, HP["n_coll"], seed=HP["seed"] + ep)

    loss = train_step(pinn, t_subset, zeta, omega0_rad_s, time_scale_s)
    l = float(loss.numpy())

    # suivi + best
    if l + 1e-12 < best_loss:
        best_loss = l
        pat = 0
        pinn.save(best_path, include_optimizer=False)
        tag = " (save)"
    else:
        pat += 1
        tag = ""

    if ep % 50 == 0 or ep == 1:
        print(f"Epoch {ep:4d}/{EPOCHS}  L_phys={l:.6e}  best={best_loss:.6e}{tag}")

    if pat >= PATIENCE:
        print(f"[EarlyStop] patience={PATIENCE} atteinte à l’epoch {ep}.")
        break

print(f"[BEST] L_phys_min={best_loss:.6e} | modèle sauvegardé -> {best_path}")

# --- 3.9  Petit historique (léger) ---
hist_path = ART_DIR / "pinn_u_only_train_history_v2_2.json"
with open(hist_path, "w") as f:
    json.dump({
        "when": stamp,
        "L": L,
        "hp": {**HP, "lr": HP["lr"]},
        "phys": {"zeta": zeta, "omega0_rad_s": omega0_rad_s, "time_scale_s": time_scale_s},
        "best_loss": best_loss,
        "best_ckpt": str(best_path)
    }, f, indent=2)

print(f"[SAVE] historique -> {hist_path}")
print("Cellule 3 terminée ✅")


[PHYS] zeta=0.0100, omega0=25.1327 rad/s, time_scale_s=655.350000
[HP]   n_coll=2048, widths=[64, 64, 64], act=tanh, lr=1.0e-03, epochs=2000, patience=200
Model: "PINN_u_of_t"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 t_norm (InputLayer)         [(None, 1)]               0         
                                                                 
 mlp_1 (Dense)               (None, 64)                128       
                                                                 
 mlp_2 (Dense)               (None, 64)                4160      
                                                                 
 mlp_3 (Dense)               (None, 64)                4160      
                                                                 
 u (Dense)                   (None, 1)                 65        
                                                                 
Total params: 8,513
Trainable pa

In [6]:
# ============================================================
# Cellule 4 — Reprise + entraînement hybride (PINN_2.3 corrigée)
#  - Charge le meilleur checkpoint disponible
#  - Définit t_norm_full (plein domaine)
#  - Défs physiques + dérivées robustes (temps rescalé si besoin)
#  - Entraînement avec L_phys (+ L_data optionnel, pondéré)
#  - Exporte historique + figure + checkpoint
# ============================================================

from __future__ import annotations
import os, json, time, math, datetime
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# --- 4.0 Chemins & artefacts ---
PROJ_DIR = Path("/home/a.riyahi/spinn_project")
OUT_DIR  = PROJ_DIR / "outputs"
ART_DIR  = OUT_DIR / "artifacts"
FIG_DIR  = OUT_DIR / "figs"
MODEL_DIR = PROJ_DIR / "models"  # FIX: on force un seul répertoire de checkpoints

for d in [OUT_DIR, ART_DIR, FIG_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TAG = "PINN_2_3"
NOW_STR = time.strftime("%Y%m%d_%H%M%S")

# --- 4.1 Charger config physique & cache (pour connaître L) ---
phys_cfg_path = ART_DIR / "pinn_phys_config_v2_2.json"   # compat v2.2 conservée
with open(phys_cfg_path, "r") as f:
    PHYS_CFG = json.load(f)

# valeurs sûres + typage
omega0 = float(PHYS_CFG.get("omega0_rad_s", 2*math.pi*4.0))
zeta   = float(PHYS_CFG.get("zeta", 0.01))
ts     = PHYS_CFG.get("time_scale_s", None)
time_scale_s = float(ts) if (ts is not None) else 0.0

# dataset cache (pour L et éventuellement construire un u_meas)
ds = np.load(OUT_DIR / "cache" / "pinn2_2_dataset_cache.npz", allow_pickle=False)  # compat v2.2
L  = int(ds["X_train"].shape[1])   # 65536 attendu

# --- 4.2 Grille temporelle normalisée complète (L,1) ---
t_norm_full = tf.reshape(
    tf.linspace(tf.constant(0.0, tf.float32), tf.constant(1.0, tf.float32), L),
    (-1, 1)
)

# --- 4.3 Hyperparamètres entraînement ---
HP = {
    "seed":        int(PHYS_CFG.get("seed", 42)),
    "n_coll":      int(PHYS_CFG.get("n_coll", 2048)),
    "widths":      list(PHYS_CFG.get("mlp_widths", [64,64,64])),
    "activation":  str(PHYS_CFG.get("activation", "tanh")),
    "lr":          float(PHYS_CFG.get("lr_cell4", 1e-3)),
    "min_lr":      float(PHYS_CFG.get("min_lr_cell4", 1e-5)),
    "epochs":      int(PHYS_CFG.get("epochs_cell4", 1500)),
    "patience":    int(PHYS_CFG.get("patience_cell4", 200)),
    "batch_data":  int(PHYS_CFG.get("batch_data", 1024)),
    # pondérations pertes
    "w_phys":      float(PHYS_CFG.get("w_phys", 1.0)),
    "w_data":      float(PHYS_CFG.get("w_data", 0.0)),   # =0.0 -> physique pur
}
tf.keras.utils.set_random_seed(HP["seed"])

print("[PHYS] zeta=%.4f, omega0=%.4f rad/s, time_scale_s=%s" %
      (zeta, omega0, f"{time_scale_s:.6f}" if time_scale_s>0 else "None"))
print("[HP]   n_coll=%d, widths=%s, act=%s, lr=%.1e→%.1e, epochs=%d, patience=%d, w_phys=%.2f, w_data=%.2f" %
      (HP["n_coll"], HP["widths"], HP["activation"], HP["lr"], HP["min_lr"],
       HP["epochs"], HP["patience"], HP["w_phys"], HP["w_data"]))

# --- 4.4 Défs physiques & dérivées (autonomes) ---
@tf.function
def h_phys_sdof(u, u_dot, u_2dot, zeta, omega0):
    z = tf.cast(zeta, tf.float32)
    w = tf.cast(omega0, tf.float32)
    return u_2dot + 2.0*z*w*u_dot + (w*w)*u

@tf.function
def derivatives_from_model(model, t, time_scale_s: float):
    t = tf.cast(t, tf.float32)   # (B,1)
    # FIX: inference stable (pas training)
    with tf.GradientTape(persistent=True) as g2:
        g2.watch(t)
        with tf.GradientTape() as g1:
            g1.watch(t)
            u = model(t, training=False)     # FIX: False → pas de dropout/BN
        u_dot = g1.gradient(u, t)            # du/dt_norm
    u_2dot = g2.gradient(u_dot, t)           # d2u/dt_norm2
    del g1, g2
    if time_scale_s and time_scale_s > 0.0:
        invT = tf.constant(1.0/time_scale_s, tf.float32)
        u_dot  = u_dot  * invT
        u_2dot = u_2dot * (invT**2)
    return u, u_dot, u_2dot

@tf.function
def compute_L_phys(model, t_subset):
    u, u_dot, u_2dot = derivatives_from_model(model, t_subset, time_scale_s)
    r = h_phys_sdof(u, u_dot, u_2dot, zeta, omega0)
    return tf.reduce_mean(tf.square(r))

# --- 4.5 Data-term optionnel (u_meas aligné sur t_norm_full) ---
u_meas = None
# Exemple pour tester rapidement :
# u_meas = tf.convert_to_tensor(ds["X_train"][0, :, 0:1], tf.float32)

# --- 4.6 Samplers (collocation & data) ---
rng = tf.random.Generator.from_seed(HP["seed"])

@tf.function
def sample_collocation(nc: tf.Tensor):  # nc: int32 scalaire
    idx = rng.uniform(shape=(nc,), minval=0, maxval=L, dtype=tf.int32)
    return tf.gather(t_norm_full, idx)

@tf.function
def sample_data_batch(bs: tf.Tensor):
    if (u_meas is None) or (HP["w_data"] <= 0.0):
        return tf.constant([], tf.float32), tf.constant([], tf.float32)
    idx = rng.uniform(shape=(bs,), minval=0, maxval=L, dtype=tf.int32)
    return tf.gather(t_norm_full, idx), tf.gather(u_meas, idx)

# --- 4.7 Chargement ckpt ou construction modèle ---
def build_pinn(widths=(64,64,64), act="tanh"):
    from tensorflow.keras import layers as L, models as KM
    inp = L.Input(shape=(1,), name="t_norm")
    x = inp
    for i, w in enumerate(widths, 1):
        x = L.Dense(w, activation=act, name=f"mlp_{i}")(x)
    out = L.Dense(1, activation=None, name="u")(x)  # sortie scalaire (None,1)
    return KM.Model(inp, out, name="PINN_u_of_t")

def find_best_ckpt():
    patterns = ["PINN_2_3_best_*.h5", "PINN_2_2_best_*.h5", "pinn_u_only_best_*.h5", "*best*.h5"]
    cands = []
    # On cherche PRIORITAIREMENT dans MODEL_DIR, puis fallback OUT_DIR/models si existait
    for d in [MODEL_DIR, OUT_DIR / "models"]:
        for pat in patterns:
            cands += list(d.glob(pat))
    if not cands:
        return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]

best_ckpt = find_best_ckpt()
if best_ckpt and best_ckpt.exists():
    try:
        pinn = tf.keras.models.load_model(best_ckpt, compile=False)
        print(f"[LOAD] Meilleur checkpoint chargé: {best_ckpt}")
    except Exception as e:
        print(f"[WARN] Échec chargement {best_ckpt} ({e}), nouveau modèle.")
        pinn = build_pinn(HP["widths"], HP["activation"])
else:
    print("[INFO] Aucun checkpoint trouvé — nouveau modèle.")
    pinn = build_pinn(HP["widths"], HP["activation"])

pinn.summary()

# --- 4.8 Pertes & optimisation ---
@tf.function
def compute_losses(model, t_coll, t_data, u_data, w_phys: tf.Tensor, w_data: tf.Tensor):
    Lp = compute_L_phys(model, t_coll)
    if (tf.shape(t_data)[0] > 0) and (tf.shape(u_data)[0] > 0) and (w_data > 0):
        u_pred = model(t_data, training=True)  # OK: le head peut être en mode train ici
        if tf.shape(u_pred)[-1] > 1:
            u_pred = u_pred[:, :1]
        Ld = tf.reduce_mean(tf.square(u_pred - u_data))
    else:
        Ld = tf.constant(0.0, tf.float32)
    Ltot = w_phys*Lp + w_data*Ld
    return Ltot, Lp, Ld

optimizer = tf.keras.optimizers.Adam(learning_rate=HP["lr"])

def cosine_lr(step, total_steps, lr0=HP["lr"], lr_min=HP["min_lr"]):
    pi = tf.constant(np.pi, tf.float32)
    ratio = tf.cast(step, tf.float32)/tf.cast(tf.maximum(total_steps,1), tf.float32)
    return lr_min + 0.5*(lr0 - lr_min)*(1+tf.cos(pi*ratio))

best_val = np.inf
no_impr  = 0
history4 = {"epoch": [], "L_tot": [], "L_phys": [], "L_data": [], "lr": []}
ckpt_save_path = None

print(f"[RUN] L={L} | n_coll/epoch={HP['n_coll']} | epochs={HP['epochs']} | w_phys={HP['w_phys']} | w_data={HP['w_data']}")
for ep in range(1, HP["epochs"]+1):
    t_coll = sample_collocation(tf.constant(HP["n_coll"], tf.int32))
    t_b, u_b = sample_data_batch(tf.constant(HP["batch_data"], tf.int32))

    with tf.GradientTape() as tape:
        L_tot, L_phys, L_data = compute_losses(
            pinn, t_coll, t_b, u_b,
            tf.constant(HP["w_phys"], tf.float32),
            tf.constant(HP["w_data"], tf.float32)
        )
    grads = tape.gradient(L_tot, pinn.trainable_variables)
    optimizer.apply_gradients(zip(grads, pinn.trainable_variables))

    # scheduler cosinus
    new_lr = float(cosine_lr(ep, HP["epochs"]).numpy())
    optimizer.learning_rate.assign(new_lr)

    # suivi
    v = float(L_tot.numpy())
    history4["epoch"].append(ep)
    history4["L_tot"].append(v)
    history4["L_phys"].append(float(L_phys.numpy()))
    history4["L_data"].append(float(L_data.numpy()))
    history4["lr"].append(new_lr)

    # meilleur (sur la même métrique que train — on peut remplacer par une vraie val fixe si dispo)
    if v < best_val - 1e-9:
        best_val = v
        no_impr  = 0
        # FIX: on sauve TOUJOURS dans MODEL_DIR (pas de roulette)
        ckpt_save_path = MODEL_DIR / f"{RUN_TAG}_best_{NOW_STR}.h5"
        pinn.save(ckpt_save_path, include_optimizer=False)
        tag = "(save)"
    else:
        no_impr += 1
        tag = ""

    if ep % 25 == 0 or ep == 1:
        print(f"Epoch {ep:4d}/{HP['epochs']:4d}  L_tot={v:.6e}  L_phys={float(L_phys.numpy()):.6e}  "
              f"L_data={float(L_data.numpy()):.3e}  lr={new_lr:.2e}  best={best_val:.6e} {tag}")

    if no_impr >= HP["patience"]:
        print(f"[EARLY-STOP] pas d'amélioration pendant {HP['patience']} epochs (best L_tot={best_val:.6e})")
        break

# --- 4.9 Exports (historique + figure) ---
hist_path = ART_DIR / f"train_history_cell4_{NOW_STR}.json"
with open(hist_path, "w") as f:
    json.dump(history4, f, indent=2)
print(f"[SAVE] history -> {hist_path}")

plt.figure()
plt.plot(history4["epoch"], history4["L_phys"], label="L_phys")
if any(v > 0 for v in history4["L_data"]):
    plt.plot(history4["epoch"], history4["L_data"], label="L_data")
plt.plot(history4["epoch"], history4["L_tot"],  label="L_total", linewidth=2)
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
fig_path = FIG_DIR / f"pinn_cell4_losses_{NOW_STR}.png"
plt.tight_layout(); plt.savefig(fig_path, dpi=160); plt.close()
print(f"[SAVE] fig -> {fig_path}")

# --- 4.10 Résumé final ---
summary = {
    "best_L_total": best_val,
    "epochs_run": len(history4["epoch"]),
    "best_ckpt": str(ckpt_save_path) if ckpt_save_path else "n/a",
    "used_data_term": bool((u_meas is not None) and (HP["w_data"] > 0.0)),
    "zeta": zeta, "omega0": omega0, "time_scale_s": time_scale_s,
}
print("[SUMMARY]", json.dumps(summary, indent=2))
print("Cellule 4 terminée ✅")


[PHYS] zeta=0.0100, omega0=25.1327 rad/s, time_scale_s=655.350000
[HP]   n_coll=2048, widths=[64, 64, 64], act=tanh, lr=1.0e-03→1.0e-05, epochs=1500, patience=200, w_phys=1.00, w_data=0.00
[LOAD] Meilleur checkpoint chargé: /home/a.riyahi/spinn_project/models/pinn_u_only_best_20251110_192637.h5
Model: "PINN_u_of_t"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 t_norm (InputLayer)         [(None, 1)]               0         
                                                                 
 mlp_1 (Dense)               (None, 64)                128       
                                                                 
 mlp_2 (Dense)               (None, 64)                4160      
                                                                 
 mlp_3 (Dense)               (None, 64)                4160      
                                                                 
 u (Dense)             

2025-11-10 19:26:56.831899: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'nc' with dtype int32
	 [[{{node nc}}]]


Epoch    1/1500  L_tot=6.618215e-04  L_phys=6.618215e-04  L_data=0.000e+00  lr=1.00e-03  best=6.618215e-04 (save)
Epoch   25/1500  L_tot=2.108242e+02  L_phys=2.108242e+02  L_data=0.000e+00  lr=9.99e-04  best=6.618215e-04 
Epoch   50/1500  L_tot=1.968219e+01  L_phys=1.968219e+01  L_data=0.000e+00  lr=9.97e-04  best=6.618215e-04 
Epoch   75/1500  L_tot=1.421805e-01  L_phys=1.421805e-01  L_data=0.000e+00  lr=9.94e-04  best=6.618215e-04 
Epoch  100/1500  L_tot=1.370046e-01  L_phys=1.370046e-01  L_data=0.000e+00  lr=9.89e-04  best=6.618215e-04 
Epoch  125/1500  L_tot=3.613101e-02  L_phys=3.613101e-02  L_data=0.000e+00  lr=9.83e-04  best=6.618215e-04 
Epoch  150/1500  L_tot=3.092420e-02  L_phys=3.092420e-02  L_data=0.000e+00  lr=9.76e-04  best=6.618215e-04 
Epoch  175/1500  L_tot=2.957113e-02  L_phys=2.957113e-02  L_data=0.000e+00  lr=9.67e-04  best=6.618215e-04 
Epoch  200/1500  L_tot=2.710463e-02  L_phys=2.710463e-02  L_data=0.000e+00  lr=9.57e-04  best=6.618215e-04 
[EARLY-STOP] pas d'amé

In [7]:
# === Cellule 5 — Diagnostics & tracés PINN (v2.3 corrigée) ===================
# - Recharge le meilleur ckpt PINN (priorité v2.3, fallback v2.2 / u_only)
# - Calcule u(t), u̇(t), ü(t) et le résidu h_phys = ü + 2ζω0 u̇ + ω0² u
# - Sauvegarde métriques (json), arrays (npz) et figures (u, u̇, ü, h, PSD)
# ============================================================================

from __future__ import annotations
import os, json, math, datetime, pathlib, glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

try:
    from scipy.signal import welch
except Exception:
    welch = None  # PSD fallback via numpy si indispo

print("[INFO] TensorFlow:", tf.__version__)

# ------------------------ 0) Contexte, chemins & utilitaires -----------------
PROJECT_ROOT = pathlib.Path(os.environ.get("SPINN_HOME", "/home/a.riyahi/spinn_project"))
MODELS_DIR   = PROJECT_ROOT / "models"

# Choix dynamique du dossier outputs (préférence v2.3, fallback v2.2 / générique)
CANDIDATE_OUTS = [
    PROJECT_ROOT / "outputs_v2_3",
    PROJECT_ROOT / "outputs_pinn_v2_3",
    PROJECT_ROOT / "outputs",                 # fallback (v2.2 utilisait "outputs")
]
OUT_DIR = None
for d in CANDIDATE_OUTS:
    if d.exists():
        OUT_DIR = d
        break
if OUT_DIR is None:
    OUT_DIR = CANDIDATE_OUTS[0]    # crée outputs_v2_3 si absent
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = OUT_DIR / "figs"
ART_DIR = OUT_DIR / "artifacts"
CACHE_DIR = OUT_DIR / "cache"
for d in (FIG_DIR, ART_DIR, CACHE_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

def _first_existing(*paths: pathlib.Path) -> pathlib.Path | None:
    for p in paths:
        if p is not None and pathlib.Path(p).exists():
            return pathlib.Path(p)
    return None

# ------------------------ 1) Charger config physique & longueur L ------------
# On privilégie la config v2.3, puis v2.2, sinon defaults stables.
phys_cfg_path = _first_existing(
    ART_DIR / "pinn_phys_config_v2_3.json",
    ART_DIR / "pinn_phys_config_v2_2.json",
)

PHYS_CFG = {
    "zeta": 0.01,
    "omega0_rad_s": float(2.0*math.pi*4.0),  # ~25.132741...
    "time_scale_s": 0.0,                    # 0.0 => pas de rescale
    "fs_hz": 100.0,                         # utile pour PSD (si on veut du physique)
    "L": 65536,
}
if phys_cfg_path is not None:
    try:
        with open(phys_cfg_path, "r") as f:
            cfg_loaded = json.load(f)
        PHYS_CFG.update({k: v for k, v in cfg_loaded.items() if v is not None})
        print(f"[LOAD] PHYS_CFG <- {phys_cfg_path}")
    except Exception as e:
        print(f"[WARN] Impossible de lire {phys_cfg_path} ({e}) – defaults utilisés.")

# Longueur L à partir du cache (v2.3 puis v2.2), sinon PHYS_CFG["L"]
cache_npz = _first_existing(
    CACHE_DIR / "pinn2_3_dataset_cache.npz",
    CACHE_DIR / "pinn2_2_dataset_cache.npz",
)
if cache_npz is not None:
    try:
        ds_shapes = np.load(cache_npz, allow_pickle=False)
        L = int(ds_shapes["X_train"].shape[1])
        print(f"[LOAD] Cache dataset: {cache_npz.name} | L={L}")
    except Exception as e:
        print(f"[WARN] Lecture cache {cache_npz} échouée ({e}) – L via PHYS_CFG.")
        L = int(PHYS_CFG.get("L", 65536))
else:
    L = int(PHYS_CFG.get("L", 65536))

zeta   = float(PHYS_CFG.get("zeta", 0.01))
omega0 = float(PHYS_CFG.get("omega0_rad_s", 2.0*math.pi*4.0))
time_scale_s = PHYS_CFG.get("time_scale_s", 0.0) or 0.0
fs_hz  = float(PHYS_CFG.get("fs_hz", 100.0))

# Grilles de temps
t_norm_full = tf.reshape(tf.linspace(tf.constant(0.0, tf.float32),
                                     tf.constant(1.0, tf.float32), L), (-1, 1))
t_phys = (np.linspace(0.0, time_scale_s, L, dtype=np.float32)
          if time_scale_s > 0.0 else
          np.linspace(0.0, 1.0, L, dtype=np.float32))

# ------------------------ 2) Trouver & charger le meilleur checkpoint --------
def _find_best_ckpt() -> pathlib.Path | None:
    patterns = [
        "PINN_2_3_best_*.h5",          # priorité v2.3
        "pinn_u_only_best_*.h5",       # pré-entraînement physique
        "PINN_2_2_best_*.h5",          # fallback v2.2
        "*best*.h5",                    # filet large
    ]
    cands = []
    for pat in patterns:
        cands.extend(sorted(MODELS_DIR.glob(pat), key=lambda p: p.stat().st_mtime, reverse=True))
    return cands[0] if cands else None

best_ckpt = _find_best_ckpt()
if best_ckpt is None:
    print("[INFO] Aucun checkpoint trouvé – construis un PINN minimal pour illustrer.")
    inp = tf.keras.Input(shape=(1,), name="t_norm")
    x = tf.keras.layers.Dense(64, activation='tanh', name="mlp_1")(inp)
    x = tf.keras.layers.Dense(64, activation='tanh', name="mlp_2")(x)
    x = tf.keras.layers.Dense(64, activation='tanh', name="mlp_3")(x)
    out = tf.keras.layers.Dense(1, name="u")(x)
    pinn = tf.keras.Model(inp, out, name="PINN_u_of_t")
    best_ckpt_str = "none"
else:
    pinn = tf.keras.models.load_model(best_ckpt, compile=False)
    best_ckpt_str = str(best_ckpt)
    print(f"[LOAD] Best checkpoint: {best_ckpt_str}")

# ------------------------ 3) Dérivées & résidu (batchées & rescalées) -------
def eval_derivatives(model: tf.keras.Model,
                     t: tf.Tensor,
                     Tphys: float,
                     batch: int = 32768) -> tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
    """Calcule u, u̇, ü sur tout t via mini-batches (faible RAM) et rescale si Tphys>0."""
    us, uds, u2ds = [], [], []
    n = int(t.shape[0])
    invT  = 1.0/float(Tphys) if Tphys and Tphys > 0.0 else 1.0
    invT2 = invT*invT
    for i in range(0, n, batch):
        tb = tf.cast(t[i:i+batch], tf.float32)
        with tf.GradientTape(persistent=True) as g1:
            g1.watch(tb)
            with tf.GradientTape(persistent=True) as g2:
                g2.watch(tb)
                u = model(tb, training=False)            # (B,1)
            udot = g2.gradient(u, tb)                    # du/dt_norm
        u2dot = g1.gradient(udot, tb)                    # d2u/dt_norm2
        del g2, g1
        if Tphys and Tphys > 0.0:
            udot  = udot * invT
            u2dot = u2dot * invT2
        us.append(u); uds.append(udot); u2ds.append(u2dot)
    return tf.concat(us, 0), tf.concat(uds, 0), tf.concat(u2ds, 0)

u, udot, u2dot = eval_derivatives(pinn, t_norm_full, time_scale_s, batch=32768)

# Résidu physique: h = ü + 2ζω0 u̇ + ω0² u
z = tf.constant(zeta,   tf.float32)
w = tf.constant(omega0, tf.float32)
h = u2dot + 2.0*z*w*udot + (w*w)*u

# ------------------------ 4) Métriques & exports artefacts -------------------
u_np   = u.numpy().reshape(-1)
ud_np  = udot.numpy().reshape(-1)
u2d_np = u2dot.numpy().reshape(-1)
h_np   = h.numpy().reshape(-1)

L2_mean = float(np.mean(h_np**2))
L1_mean = float(np.mean(np.abs(h_np)))
Linf    = float(np.max(np.abs(h_np)))
abs_h_p95 = float(np.percentile(np.abs(h_np), 95.0))

res_summary = {
    "L2_mean": L2_mean,
    "L1_mean": L1_mean,
    "Linf": Linf,
    "abs_h_p95": abs_h_p95,
    "zeta": zeta,
    "omega0": omega0,
    "time_scale_s": float(time_scale_s),
}
print("[RESIDUALS]", json.dumps(res_summary, indent=2))

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# npz compact
npz_path = ART_DIR / f"pinn2_3_diagnostics_{stamp}.npz"
np.savez_compressed(
    npz_path,
    t_phys=t_phys.astype(np.float32),
    t_norm=t_norm_full.numpy().squeeze().astype(np.float32),
    u=u_np.astype(np.float32),
    u_dot=ud_np.astype(np.float32),
    u_ddot=u2d_np.astype(np.float32),
    h=h_np.astype(np.float32),
    zeta=np.float32(zeta),
    omega0=np.float32(omega0),
    time_scale_s=np.float32(time_scale_s),
    best_ckpt=np.string_(best_ckpt_str),
)
print(f"[SAVE] npz -> {npz_path}")

# json résumé
diag_json = {
    "when": stamp,
    "best_ckpt": best_ckpt_str,
    "metrics": res_summary,
    "L": int(L),
    "fs_hz": float(fs_hz),
}
with open(ART_DIR / "diagnostics_summary_v2_3.json", "w") as f:
    json.dump(diag_json, f, indent=2)
print("[SAVE] diagnostics_summary_v2_3.json")

# ------------------------ 5) Figures (u, u̇, ü, h) --------------------------
def _plot_and_save(x, y, xlabel, ylabel, title, path, xlim_n=None):
    plt.figure(figsize=(10,3))
    plt.plot(x, y, linewidth=1.1)
    if xlim_n is not None and xlim_n < len(x):
        plt.xlim(x[0], x[xlim_n-1])
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(title); plt.grid(True, alpha=0.25)
    plt.tight_layout(); plt.savefig(path, dpi=160); plt.close()
    print(f"[SAVE] fig -> {path}")

_xlim = min(5_000, L)
xlab = "t [s]" if time_scale_s > 0.0 else "t_norm"

_plot_and_save(t_phys, u_np,   xlab, "u",   "PINN — u(t)",   FIG_DIR / f"pinn_u_{stamp}.png",    xlim_n=_xlim)
_plot_and_save(t_phys, ud_np,  xlab, "u̇",  "PINN — u̇(t)",  FIG_DIR / f"pinn_udot_{stamp}.png", xlim_n=_xlim)
_plot_and_save(t_phys, u2d_np, xlab, "ü",  "PINN — ü(t)",  FIG_DIR / f"pinn_uddot_{stamp}.png", xlim_n=_xlim)
_plot_and_save(t_phys, h_np,   xlab, "h",   "Résidu h(t) = ü + 2ζω0 u̇ + ω0² u",
               FIG_DIR / f"pinn_residual_{stamp}.png", xlim_n=_xlim)

# ------------------------ 6) PSD (Welch si possible, sinon FFT) --------------
def _plot_psd(u_sig: np.ndarray, fs: float, title: str, out_path: pathlib.Path):
    if welch is not None and fs > 0:
        f_u, Pxx_u = welch(u_sig, fs=fs, nperseg=min(8192, len(u_sig)))
        plt.figure(figsize=(10,3))
        plt.semilogy(f_u, Pxx_u + 1e-18)
        plt.xlabel("Fréquence [Hz]"); plt.ylabel("PSD")
        plt.title(title + " (Welch)")
        plt.grid(True, which="both", alpha=0.25)
        plt.tight_layout(); plt.savefig(out_path, dpi=160); plt.close()
        print(f"[SAVE] fig -> {out_path}")
    else:
        # PSD sur t_norm (fréquence en bins)
        u_c = u_sig - np.mean(u_sig)
        U = np.fft.rfft(u_c)
        PSD = (np.abs(U)**2) / max(1, len(U))
        f_bins = np.fft.rfftfreq(len(u_c), d=1.0/len(u_c))
        plt.figure(figsize=(10,3))
        plt.semilogy(f_bins, PSD + 1e-18)
        plt.xlabel("fréquence (bins t_norm)"); plt.ylabel("PSD |U|^2")
        plt.title(title + " (rFFT)")
        plt.grid(True, which="both", alpha=0.25)
        plt.tight_layout(); plt.savefig(out_path, dpi=160); plt.close()
        print(f"[SAVE] fig -> {out_path}")

_plot_psd(u_np, fs_hz if time_scale_s > 0.0 else 0.0,
          "PINN — PSD(u)", FIG_DIR / f"pinn_psd_{stamp}.png")

print("Cellule 5 terminée ✅ (v2.3)")


[INFO] TensorFlow: 2.12.0
[LOAD] Best checkpoint: /home/a.riyahi/spinn_project/models/PINN_2_3_best_20251110_192656.h5
[RESIDUALS] {
  "L2_mean": 5146.3525390625,
  "L1_mean": 65.81021118164062,
  "Linf": 113.27259063720703,
  "abs_h_p95": 108.95991516113281,
  "zeta": 0.01,
  "omega0": 25.132741228718345,
  "time_scale_s": 0.0
}
[SAVE] npz -> /home/a.riyahi/spinn_project/outputs_pinn_v2_3/artifacts/pinn2_3_diagnostics_20251110_192706.npz
[SAVE] diagnostics_summary_v2_3.json
[SAVE] fig -> /home/a.riyahi/spinn_project/outputs_pinn_v2_3/figs/pinn_u_20251110_192706.png
[SAVE] fig -> /home/a.riyahi/spinn_project/outputs_pinn_v2_3/figs/pinn_udot_20251110_192706.png
[SAVE] fig -> /home/a.riyahi/spinn_project/outputs_pinn_v2_3/figs/pinn_uddot_20251110_192706.png
[SAVE] fig -> /home/a.riyahi/spinn_project/outputs_pinn_v2_3/figs/pinn_residual_20251110_192706.png
[SAVE] fig -> /home/a.riyahi/spinn_project/outputs_pinn_v2_3/figs/pinn_psd_20251110_192706.png
Cellule 5 terminée ✅ (v2.3)


In [8]:
# === Cellule 6 — Fine‑tuning PINN (phys + data) robuste ======================
# Objectif : reprendre le meilleur ckpt (Cell.3/4), ajouter un terme data léger,
#            entraîner avec curriculum w_data (0 -> W_DATA_MAX), early-stop,
#            et sauvegarder ckpt + historique.
# ============================================================================

import os, json, datetime, math, pathlib
import numpy as np
import tensorflow as tf

print("[INFO] TensorFlow:", tf.__version__)

# ---------------------------------------------------------------------
# 0) Contexte & filets de sécurité
# ---------------------------------------------------------------------
PROJ_DIR  = pathlib.Path(os.environ.get("SPINN_HOME", "/home/a.riyahi/spinn_project"))
OUT_DIR   = PROJ_DIR / "outputs"
ART_DIR   = OUT_DIR / "artifacts"
FIG_DIR   = OUT_DIR / "figs"
MODEL_DIR = PROJ_DIR / "models"
for d in (OUT_DIR, ART_DIR, FIG_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEFAULT_L = 65536

# PHYS_CFG : combler les trous + typer proprement (évite les None -> float())
if 'PHYS_CFG' not in globals() or PHYS_CFG is None:
    PHYS_CFG = {}
PHYS_CFG.setdefault("zeta", 0.01)
PHYS_CFG.setdefault("omega0_rad_s", float(2.0*math.pi*4.0))  # ~ 25.1327 rad/s
# Si time_scale_s n'est pas défini, on force 0.0 (pas de rescale) — évite TypeError
if PHYS_CFG.get("time_scale_s", None) is None:
    PHYS_CFG["time_scale_s"] = 0.0
PHYS_CFG.setdefault("n_collocation", 2048)
PHYS_CFG.setdefault("seed", 42)
PHYS_CFG.setdefault("L", DEFAULT_L)

zeta   = tf.constant(float(PHYS_CFG["zeta"]), dtype=tf.float32)
omega0 = tf.constant(float(PHYS_CFG["omega0_rad_s"]), dtype=tf.float32)
Tphys  = tf.constant(float(PHYS_CFG.get("time_scale_s", 0.0)), dtype=tf.float32)

# t_norm_full : (L,1) sur [0,1]
if 't_norm_full' not in globals() or t_norm_full is None:
    L = int(PHYS_CFG.get("L", DEFAULT_L))
    t_norm_full = tf.linspace(0.0, 1.0, L)[:, None]
else:
    t_norm_full = tf.convert_to_tensor(t_norm_full, dtype=tf.float32)
    L = int(t_norm_full.shape[0])

# u_meas : optionnel. Si absent/None, on génère un fallback neutre (zéros)
# (Tu peux brancher ici un vrai signal mesuré aligné sur t_norm_full)
if 'u_meas' not in globals() or u_meas is None:
    try:
        # Fallback "intelligent" : 1er signal du cache si présent
        ds = np.load(OUT_DIR / "cache" / "pinn2_2_dataset_cache.npz", allow_pickle=False)
        u_meas = tf.convert_to_tensor(ds["X_train"][0, :, 0:1], tf.float32)
        if int(u_meas.shape[0]) != L:
            # on remet à L (pad/trim simple)
            u_meas = tf.reshape(u_meas, (-1,1))
            if u_meas.shape[0] < L:
                pad = tf.fill([L - int(u_meas.shape[0]), 1], u_meas[-1,0])
                u_meas = tf.concat([u_meas, pad], axis=0)
            else:
                u_meas = u_meas[:L, :]
        print("[INFO] u_meas chargé depuis cache (fallback).")
    except Exception:
        u_meas = tf.zeros((L, 1), dtype=tf.float32)
        print("[WARN] u_meas indisponible -> zeros (fallback).")
else:
    u_meas = tf.cast(u_meas, tf.float32)
    if u_meas.shape[-1] != 1:
        u_meas = u_meas[..., None]
    if int(u_meas.shape[0]) != L:
        raise ValueError(f"u_meas doit avoir {L} échantillons, trouvé {int(u_meas.shape[0])}.")

# Chargement du meilleur ckpt disponible
best_ckpt = None
cands = sorted(MODEL_DIR.glob("PINN_2_2_best_*.h5"))[::-1]
if cands:
    best_ckpt = str(cands[0])

if best_ckpt:
    try:
        pinn = tf.keras.models.load_model(best_ckpt, compile=False)
        print(f"[LOAD] Best ckpt: {best_ckpt}")
    except Exception as e:
        print(f"[WARN] Échec chargement {best_ckpt} -> {e}")
        # filet de secours : petit MLP
        inp = tf.keras.Input(shape=(1,), name="t_norm")
        x = tf.keras.layers.Dense(64, activation='tanh', name="mlp_1")(inp)
        x = tf.keras.layers.Dense(64, activation='tanh', name="mlp_2")(x)
        x = tf.keras.layers.Dense(64, activation='tanh', name="mlp_3")(x)
        out = tf.keras.layers.Dense(1, name="u")(x)
        pinn = tf.keras.Model(inp, out, name="PINN_u_of_t")
        print("[INFO] Modèle PINN reconstruit (secours).")
else:
    # Si aucun ckpt, on part d’un MLP propre
    inp = tf.keras.Input(shape=(1,), name="t_norm")
    x = tf.keras.layers.Dense(64, activation='tanh', name="mlp_1")(inp)
    x = tf.keras.layers.Dense(64, activation='tanh', name="mlp_2")(x)
    x = tf.keras.layers.Dense(64, activation='tanh', name="mlp_3")(x)
    out = tf.keras.layers.Dense(1, name="u")(x)
    pinn = tf.keras.Model(inp, out, name="PINN_u_of_t")
    print("[INFO] Aucun checkpoint trouvé — modèle reconstruit.")

# ---------------------------------------------------------------------
# 1) Dérivées & physique (tf.function avec signatures fixes)
# ---------------------------------------------------------------------
@tf.function(reduce_retracing=True)
def _derivatives_from_model(model, t):  # t: (B,1) float32 in [0,1]
    with tf.GradientTape(persistent=True) as g1:
        g1.watch(t)
        with tf.GradientTape(persistent=True) as g2:
            g2.watch(t)
            u = model(t, training=False)
        udot = g2.gradient(u, t)  # du/dt_norm
    u2dot = g1.gradient(udot, t)  # d2u/dt_norm2
    del g1, g2

    # Rescaling si temps physique connu
    if Tphys > 0:
        invT  = 1.0 / Tphys
        invT2 = invT * invT
        udot  = udot * invT
        u2dot = u2dot * invT2
    return u, udot, u2dot

@tf.function(reduce_retracing=True)
def h_phys_sdof(u, udot, u2dot, zeta, omega0):
    # h = u¨ + 2 ζ ω0 u˙ + ω0^2 u
    return u2dot + 2.0*zeta*omega0*udot + (omega0*omega0)*u

@tf.function(input_signature=[tf.TensorSpec(shape=[None,1], dtype=tf.float32)])
def L_phys_on_batch(t_coll):
    u, udot, u2dot = _derivatives_from_model(pinn, t_coll)
    h = h_phys_sdof(u, udot, u2dot, zeta, omega0)
    return tf.reduce_mean(tf.square(h))

mse = tf.keras.losses.MeanSquaredError(reduction='sum_over_batch_size')

@tf.function(input_signature=[
    tf.TensorSpec([None,1], tf.float32),  # t_data
    tf.TensorSpec([None,1], tf.float32)   # u_ref
])
def L_data_on_batch(t_data, u_ref):
    u_pred, _, _ = _derivatives_from_model(pinn, t_data)
    return mse(u_ref, u_pred)

# ---------------------------------------------------------------------
# 2) RNG & jeux de validation figés (pas de retracing aléatoire)
# ---------------------------------------------------------------------
rng = tf.random.Generator.from_seed(int(PHYS_CFG.get("seed", 42)))

@tf.function(input_signature=[
    tf.TensorSpec([], tf.int32),   # bs collocation
    tf.TensorSpec([], tf.int32)    # bs data
])
def sample_indices(bs_coll, bs_data):
    # RNG TensorFlow côté graph -> pas de retracing Python
    idx_coll = rng.uniform(shape=(bs_coll,), maxval=L, dtype=tf.int32)
    idx_data = rng.uniform(shape=(bs_data,), maxval=L, dtype=tf.int32)
    return idx_coll, idx_data

# Validation (indices fixes)
BS_VAL_COLL = 4096
BS_VAL_DATA = 8192
idx_val_coll = tf.random.stateless_uniform((BS_VAL_COLL,), seed=[123, 456], maxval=L, dtype=tf.int32)
idx_val_data = tf.random.stateless_uniform((BS_VAL_DATA,), seed=[789, 101], maxval=L, dtype=tf.int32)
t_val_coll = tf.gather(t_norm_full, idx_val_coll)
t_val_data = tf.gather(t_norm_full, idx_val_data)
u_val_data = tf.gather(u_meas,      idx_val_data)

# ---------------------------------------------------------------------
# 3) Optimiseur + train_step (signatures fixes) + scheduler
# ---------------------------------------------------------------------
init_lr = 5e-4
lr_var  = tf.Variable(init_lr, dtype=tf.float32, trainable=False)
opt = tf.keras.optimizers.Adam(learning_rate=lr_var)

@tf.function(input_signature=[
    tf.TensorSpec([None,1], tf.float32),  # t_coll
    tf.TensorSpec([None,1], tf.float32),  # t_data
    tf.TensorSpec([None,1], tf.float32),  # u_data
    tf.TensorSpec([], tf.float32),        # w_phys
    tf.TensorSpec([], tf.float32)         # w_data
])
def train_step(t_coll, t_data, u_data, w_phys, w_data):
    with tf.GradientTape() as tape:
        Lp = L_phys_on_batch(t_coll)
        Ld = L_data_on_batch(t_data, u_data)
        Ltot = w_phys*Lp + w_data*Ld
    grads = tape.gradient(Ltot, pinn.trainable_variables)
    opt.apply_gradients(zip(grads, pinn.trainable_variables))
    return Ltot, Lp, Ld

@tf.function(input_signature=[])
def lr_decay():
    lr_var.assign(lr_var * tf.constant(0.999, tf.float32))  # décroissance douce

# ---------------------------------------------------------------------
# 4) Boucle d’entraînement — curriculum sur w_data (0 -> W_DATA_MAX)
# ---------------------------------------------------------------------
EPOCHS        = 1200            # tu peux ajuster
BS_COLL       = int(PHYS_CFG.get("n_collocation", 2048))
BS_DATA       = 4096
W_PHYS        = tf.constant(1.0,   tf.float32)
W_DATA_MAX    = tf.constant(0.05,  tf.float32)  # 5% pour ne pas écraser la physique
PATIENCE      = 120

best_val      = np.inf
stamp_ckpt    = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
best_path     = str(MODEL_DIR / f"PINN_2_2_best_{stamp_ckpt}.h5")
no_improve    = 0

print(f"[RUN] epochs={EPOCHS} | n_coll={BS_COLL} | bs_data={BS_DATA} | w_data_max={float(W_DATA_MAX):.3f} | lr0={init_lr:.1e}")

hist = []

for epoch in range(1, EPOCHS+1):
    # Poids data progressif (linéaire)
    w_data = W_DATA_MAX * tf.cast(epoch, tf.float32) / tf.cast(EPOCHS, tf.float32)

    # échantillonnage
    idx_coll, idx_data = sample_indices(tf.constant(BS_COLL, tf.int32),
                                        tf.constant(BS_DATA, tf.int32))
    t_coll = tf.gather(t_norm_full, idx_coll)
    t_data = tf.gather(t_norm_full, idx_data)
    u_data = tf.gather(u_meas,      idx_data)

    # step
    Ltot, Lp, Ld = train_step(t_coll, t_data, u_data, W_PHYS, w_data)

    # validation (fixe)
    Lp_val    = L_phys_on_batch(t_val_coll)
    Ld_val    = L_data_on_batch(t_val_data, u_val_data)
    Ltot_val  = W_PHYS*Lp_val + w_data*Ld_val

    # scheduler
    lr_decay()

    # suivi
    rec = {
        "epoch": epoch,
        "lr": float(lr_var.numpy()),
        "w_data": float(w_data.numpy()),
        "train": {"Ltot": float(Ltot.numpy()), "Lp": float(Lp.numpy()), "Ld": float(Ld.numpy())},
        "val":   {"Ltot": float(Ltot_val.numpy()), "Lp": float(Lp_val.numpy()), "Ld": float(Ld_val.numpy())}
    }
    hist.append(rec)

    if epoch % 25 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d}/{EPOCHS} | TR: Ltot={rec['train']['Ltot']:.3e} Lp={rec['train']['Lp']:.3e} Ld={rec['train']['Ld']:.3e} "
              f"| VA: Ltot={rec['val']['Ltot']:.3e} Lp={rec['val']['Lp']:.3e} Ld={rec['val']['Ld']:.3e} "
              f"| lr={rec['lr']:.2e} w_data={rec['w_data']:.3f}")

    # early stop + best ckpt
    if rec['val']['Ltot'] + 1e-12 < best_val:
        best_val   = rec['val']['Ltot']
        no_improve = 0
        pinn.save(best_path)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"[EARLY-STOP] Pas d'amélioration {PATIENCE} epochs (best={best_val:.3e})")
            break

# Sauvegardes: historique & méta
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
hist_path = OUT_DIR / "artifacts" / f"train_history_cell6_{stamp}.json"
hist_path.parent.mkdir(parents=True, exist_ok=True)
with open(hist_path, "w") as f:
    json.dump({
        "config": {
            "epochs": EPOCHS, "bs_coll": BS_COLL, "bs_data": BS_DATA,
            "w_phys": 1.0, "w_data_max": float(W_DATA_MAX.numpy()),
            "init_lr": init_lr, "patience": PATIENCE,
            "zeta": float(zeta.numpy()), "omega0": float(omega0.numpy()),
            "time_scale_s": float(Tphys.numpy())
        },
        "best_val": best_val,
        "best_ckpt": best_path,
        "history": hist
    }, f, indent=2)

print(f"[SAVE] history -> {hist_path}")
print(f"[SAVE] best_ckpt -> {best_path}")
print("Cellule 6 terminée ✅")


[INFO] TensorFlow: 2.12.0
[INFO] u_meas chargé depuis cache (fallback).
[LOAD] Best ckpt: /home/a.riyahi/spinn_project/models/PINN_2_2_best_20251110_004332.h5
[RUN] epochs=1200 | n_coll=2048 | bs_data=4096 | w_data_max=0.050 | lr0=5.0e-04


2025-11-10 19:27:08.257072: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'bs_coll' with dtype int32
	 [[{{node bs_coll}}]]
2025-11-10 19:27:08.269764: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'bs_data' with dtype int32
	 [[{{node bs_data}}]]
2025-11-10 19:27:08.967134: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/PartitionedCall_grad/PartitionedCall' with dtype float and shape [?,1]
	 [[{{node gradients/Partitioned

Epoch    1/1200 | TR: Ltot=5.208e-03 Lp=5.196e-03 Ld=2.702e-01 | VA: Ltot=1.773e+03 Lp=1.773e+03 Ld=3.376e-01 | lr=5.00e-04 w_data=0.000
Epoch   25/1200 | TR: Ltot=5.912e+00 Lp=5.912e+00 Ld=2.738e-01 | VA: Ltot=7.770e+00 Lp=7.769e+00 Ld=2.659e-01 | lr=4.88e-04 w_data=0.001
Epoch   50/1200 | TR: Ltot=7.543e+00 Lp=7.542e+00 Ld=2.664e-01 | VA: Ltot=7.122e+00 Lp=7.122e+00 Ld=2.661e-01 | lr=4.76e-04 w_data=0.002
Epoch   75/1200 | TR: Ltot=6.150e-01 Lp=6.142e-01 Ld=2.713e-01 | VA: Ltot=4.900e-01 Lp=4.892e-01 Ld=2.711e-01 | lr=4.64e-04 w_data=0.003
Epoch  100/1200 | TR: Ltot=6.061e-03 Lp=4.930e-03 Ld=2.713e-01 | VA: Ltot=1.049e-02 Lp=9.364e-03 Ld=2.702e-01 | lr=4.52e-04 w_data=0.004
Epoch  125/1200 | TR: Ltot=3.586e-03 Lp=2.179e-03 Ld=2.701e-01 | VA: Ltot=2.480e-03 Lp=1.073e-03 Ld=2.701e-01 | lr=4.41e-04 w_data=0.005
Epoch  150/1200 | TR: Ltot=2.623e-03 Lp=9.351e-04 Ld=2.701e-01 | VA: Ltot=2.854e-03 Lp=1.166e-03 Ld=2.701e-01 | lr=4.30e-04 w_data=0.006
Epoch  175/1200 | TR: Ltot=2.861e-03 Lp=8

In [9]:
# ============================================================
# Cellule 7 — Bundle article (résumés, tableaux, figures) [corrigée]
# - Cherche diagnostics *.npz (v2.3/v2.2)
# - Si absent: reconstruit un diagnostic minimal depuis le meilleur checkpoint
# - Calcule KPIs (L2/L1/Linf/p95, pics PSD, ω0 estimée)
# - Produit figures "prêtes article" + tableau CSV
# - Écrit un JSON "paper_bundle.json" avec chemins & métriques
# ============================================================

from __future__ import annotations
import os, re, json, glob, math, datetime
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# SciPy optionnel (PSD)
try:
    from scipy.signal import welch, find_peaks
except Exception:
    welch = None
    find_peaks = None

# ---------- 0) Chemins ----------
PROJ_DIR = Path("/home/a.riyahi/spinn_project")
OUT_DIR  = PROJ_DIR / "outputs"
ART_DIR  = OUT_DIR / "artifacts"
FIG_DIR  = OUT_DIR / "figs"
MODEL_DIR= PROJ_DIR / "models"
CACHE_DIR= OUT_DIR / "cache"
for d in [OUT_DIR, ART_DIR, FIG_DIR, CACHE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# ---------- 1) Charger config physique (v2.3 ou v2.2) ----------
phys_cfg_candidates = [
    ART_DIR / "pinn_phys_config_v2_3.json",
    ART_DIR / "pinn_phys_config_v2_2.json",
]
PHYS_CFG = {}
for p in phys_cfg_candidates:
    if p.exists():
        with open(p, "r") as f:
            PHYS_CFG = json.load(f)
        break

def _get(cfg, k, default):
    v = cfg.get(k, default) if cfg else default
    return default if v is None else v

omega0 = float(_get(PHYS_CFG, "omega0_rad_s", 2*np.pi*4.0))
zeta   = float(_get(PHYS_CFG, "zeta", 0.01))
fs_hz  = float(_get(PHYS_CFG, "fs_hz", 0.0))     # si 0 -> on reste en t_norm
Tphys  = float(_get(PHYS_CFG, "time_scale_s", 0.0))  # time scale (s) ; 0 => t_norm

# ---------- 2) Utilitaires recherche ----------
def _latest(paths_or_patterns):
    cands = []
    for pat in paths_or_patterns:
        cands += glob.glob(str(pat))
    if not cands:
        return None
    cands.sort(key=lambda s: Path(s).stat().st_mtime, reverse=True)
    return Path(cands[0])

def _latest_ckpt():
    pats = [
        MODEL_DIR / "PINN_2_3_best_*.h5",
        MODEL_DIR / "PINN_2_2_best_*.h5",
        MODEL_DIR / "pinn_u_only_best_*.h5",
        MODEL_DIR / "*best*.h5",
    ]
    return _latest(pats)

# history (cell 4 / cell 7 si existante)
hist4 = _latest([ART_DIR / "train_history_cell4_*.json",
                 ART_DIR / "train_history_cell7_*.json"])

# diagnostics npz (cell 6 attendue) — motifs étendus
diag_npz = _latest([
    ART_DIR / "pinn2_3_diagnostics_*.npz",
    ART_DIR / "pinn2_2_diagnostics_*.npz",
    ART_DIR / "diagnostics_*.npz"
])
diag_json = _latest([ART_DIR / "diagnostics_summary*.json",
                     ART_DIR / "diagnostics_summary.json"])

best_ckpt = _latest_ckpt()

# ---------- 2.b) Fallback: reconstruire diagnostics si absent ----------
if (diag_npz is None) or (not diag_npz.exists()):
    print("[INFO] Aucun diagnostics NPZ trouvé — reconstruction à partir du meilleur checkpoint.")
    if best_ckpt is None or (not best_ckpt.exists()):
        raise FileNotFoundError("Aucun diagnostics NPZ ni checkpoint trouvé. Exécuter Cellule 4 (entraînement) ou Cellule 6 (diagnostics).")
    # Imports locaux pour la reconstruction
    import tensorflow as tf
    tf.get_logger().setLevel("ERROR")

    # 1) Charger le modèle
    try:
        pinn = tf.keras.models.load_model(best_ckpt, compile=False)
    except Exception as e:
        raise RuntimeError(f"Échec chargement checkpoint {best_ckpt}: {e}")

    # 2) Récupérer L (longueur) depuis cache ou via heuristique
    L = None
    cache_npz = CACHE_DIR / "pinn2_2_dataset_cache.npz"
    if cache_npz.exists():
        try:
            tmp = np.load(cache_npz, allow_pickle=False)
            L = int(tmp["X_train"].shape[1])
        except Exception:
            pass
    if (L is None) or (L <= 0):
        # fallback à 65536 (attendu pour Z24-avt)
        L = int(_get(PHYS_CFG, "L", 65536))
    # 3) Grille temporelle normalisée et conversion en t_phys si échelle fournie
    t_norm = tf.reshape(tf.linspace(tf.constant(0.0, tf.float32), tf.constant(1.0, tf.float32), L), (-1,1))
    if Tphys and Tphys > 0.0:
        t_phys = (t_norm.numpy().astype(np.float64)) * Tphys
    else:
        t_phys = t_norm.numpy().astype(np.float64)

    # 4) Dérivées via autograd + rescaling temporel (comme Cellule 4)
    z = tf.constant(zeta,  tf.float32)
    w = tf.constant(omega0,tf.float32)
    invT = tf.constant(1.0/float(Tphys), tf.float32) if (Tphys and Tphys>0.0) else None

    with tf.GradientTape(persistent=True) as g2:
        g2.watch(t_norm)
        with tf.GradientTape() as g1:
            g1.watch(t_norm)
            u = pinn(t_norm, training=False)   # (L,1)
        u_dot = g1.gradient(u, t_norm)         # du/dt_norm
    u_2dot = g2.gradient(u_dot, t_norm)        # d2u/dt_norm2
    del g1, g2

    if invT is not None:
        u_dot  = u_dot  * invT
        u_2dot = u_2dot * (invT**2)

    # 5) Résidu physique h = ü + 2ζω0 u̇ + ω0^2 u
    h = u_2dot + 2.0*z*w*u_dot + (w*w)*u

    # 6) Sauvegarde diagnostics minimal
    diag_npz = ART_DIR / f"pinn2_3_diagnostics_{STAMP}.npz"
    np.savez_compressed(
        diag_npz,
        t_phys=t_phys,         # (L,1) ou (L,) en secondes si Tphys>0, sinon t_norm
        u=u.numpy().astype(np.float64).reshape(-1),
        u_dot=u_dot.numpy().astype(np.float64).reshape(-1),
        u_ddot=u_2dot.numpy().astype(np.float64).reshape(-1),
        h=h.numpy().astype(np.float64).reshape(-1),
    )
    print(f"[BUILD] diagnostics -> {diag_npz}")

# ---------- 3) Charger diagnostics ----------
D = np.load(diag_npz, allow_pickle=False)
t_phys = D.get("t_phys", D.get("t_norm"))
u      = D["u"].astype(np.float64)
u_dot  = D.get("u_dot", None)
u_ddot = D.get("u_ddot", D.get("u2dot", None))
h      = D["h"].astype(np.float64)
L      = u.shape[0]

if t_phys is None:
    t_phys = np.linspace(0, 1, L, dtype=np.float64)
else:
    t_phys = np.asarray(t_phys, dtype=np.float64).reshape(-1)

# ---------- 4) KPIs résidu ----------
L2_mean = float(np.mean(h**2))
L1_mean = float(np.mean(np.abs(h)))
Linf    = float(np.max(np.abs(h)))
P95     = float(np.percentile(np.abs(h), 95.0))

# ---------- 5) PSD & estimation ω ----------
psd_info = {}
if welch is not None:
    if fs_hz > 0:
        fu, Puu = welch(u, fs=fs_hz, nperseg=min(8192, L))
        fh, Phh = welch(h, fs=fs_hz, nperseg=min(8192, L))
        if find_peaks is not None:
            pk_u, _ = find_peaks(Puu)
            pk_h, _ = find_peaks(Phh)
        else:
            pk_u = np.argsort(Puu)[-5:]
            pk_h = np.argsort(Phh)[-5:]
        fpk_u = float(fu[pk_u[np.argmax(Puu[pk_u])]]) if pk_u.size>0 else float(fu[np.argmax(Puu)])
        fpk_h = float(fh[pk_h[np.argmax(Phh[pk_h])]]) if pk_h.size>0 else float(fh[np.argmax(Phh)])
        omega_est = 2*np.pi*fpk_u
        psd_info = {"mode":"welch_hz","f_peak_u_hz":fpk_u,"f_peak_h_hz":fpk_h,"omega_est_rad_s":float(omega_est)}
    else:
        Fu, Puu = welch(u, fs=L, nperseg=min(8192, L))  # fs=L -> bins t_norm
        Fh, Phh = welch(h, fs=L, nperseg=min(8192, L))
        if find_peaks is not None:
            pk_u, _ = find_peaks(Puu)
        else:
            pk_u = np.argsort(Puu)[-5:]
        fpk_u = float(Fu[pk_u[np.argmax(Puu[pk_u])]]) if pk_u.size>0 else float(Fu[np.argmax(Puu)])
        omega_est = 2*np.pi*fpk_u
        psd_info = {"mode":"welch_bins","f_peak_u_bins":fpk_u,"omega_est_rad_s":float(omega_est)}
else:
    psd_info = {"mode":"none"}

# ---------- 6) Figures « prêtes article » ----------
def _style():
    plt.rcParams.update({
        "font.size": 11,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "figure.dpi": 160
    })
_style()
FIGS = {}

# (a) u(t) — zoom sur première fenêtre si long
plt.figure(figsize=(6.0, 2.6))
nwin = min(5000, L)
plt.plot(t_phys[:nwin], u[:nwin], linewidth=1.2)
plt.xlabel("t [s]" if Tphys>0 else "t_norm")
plt.ylabel("u")
plt.title("PINN — déplacement u(t)")
FIGS["u"] = str(FIG_DIR / f"paper_u_{STAMP}.png")
plt.tight_layout(); plt.savefig(FIGS["u"]); plt.close()

# (b) h(t) — même fenêtre
plt.figure(figsize=(6.0, 2.6))
plt.plot(t_phys[:nwin], h[:nwin], linewidth=1.0)
plt.xlabel("t [s]" if Tphys>0 else "t_norm")
plt.ylabel("h(t)")
plt.title("Résidu h(t) = ü + 2ζω₀u̇ + ω₀²u")
FIGS["h"] = str(FIG_DIR / f"paper_residual_{STAMP}.png")
plt.tight_layout(); plt.savefig(FIGS["h"]); plt.close()

# (c) Histogramme du résidu |h|
plt.figure(figsize=(4.5, 3.2))
plt.hist(np.abs(h), bins=80, density=True)
plt.xlabel("|h|"); plt.ylabel("densité")
plt.title("Distribution du résidu |h|")
FIGS["hist_h"] = str(FIG_DIR / f"paper_residual_hist_{STAMP}.png")
plt.tight_layout(); plt.savefig(FIGS["hist_h"]); plt.close()

# (d) PSD (u & h) si possible
if welch is not None:
    if fs_hz > 0:
        fu, Puu = welch(u, fs=fs_hz, nperseg=min(8192, L))
        fh, Phh = welch(h, fs=fs_hz, nperseg=min(8192, L))
        xlab = "Fréquence [Hz]"
    else:
        fu, Puu = welch(u, fs=L, nperseg=min(8192, L))
        fh, Phh = welch(h, fs=L, nperseg=min(8192, L))
        xlab = "Fréquence (bins t_norm)"
    plt.figure(figsize=(6.0, 3.0))
    plt.semilogy(fu, Puu + 1e-18, label="u")
    plt.semilogy(fh, Phh + 1e-18, label="h")
    plt.xlabel(xlab); plt.ylabel("PSD")
    plt.title("PSD — u et résidu h"); plt.legend()
    FIGS["psd"] = str(FIG_DIR / f"paper_psd_{STAMP}.png")
    plt.tight_layout(); plt.savefig(FIGS["psd"]); plt.close()

# (e) Courbe d’entraînement si historique dispo
HIST = {}
if hist4 is not None and hist4.exists():
    with open(hist4, "r") as f:
        HIST = json.load(f)
    if all(k in HIST for k in ("epoch","L_tot")) and len(HIST["epoch"])>0:
        plt.figure(figsize=(6.0, 3.0))
        plt.plot(HIST["epoch"], HIST["L_tot"], label="L_total", linewidth=2)
        if "L_phys" in HIST: plt.plot(HIST["epoch"], HIST["L_phys"], label="L_phys")
        if "L_data" in HIST and np.max(HIST["L_data"])>0:
            plt.plot(HIST["epoch"], HIST["L_data"], label="L_data")
        plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
        plt.title("Évolution des pertes (cell 4)")
        FIGS["train"] = str(FIG_DIR / f"paper_training_{STAMP}.png")
        plt.tight_layout(); plt.savefig(FIGS["train"]); plt.close()

# ---------- 7) Tableau CSV (KPI) ----------
CSV_PATH = ART_DIR / f"paper_kpis_{STAMP}.csv"
with open(CSV_PATH, "w") as f:
    f.write("metric,value\n")
    f.write(f"L2_mean,{L2_mean:.8e}\n")
    f.write(f"L1_mean,{L1_mean:.8e}\n")
    f.write(f"Linf,{Linf:.8e}\n")
    f.write(f"abs_h_p95,{P95:.8e}\n")
    f.write(f"zeta,{zeta}\n")
    f.write(f"omega0_rad_s,{omega0}\n")
    f.write(f"time_scale_s,{Tphys}\n")
    if "omega_est_rad_s" in psd_info:
        f.write(f"omega_est_rad_s,{psd_info['omega_est_rad_s']}\n")

# ---------- 8) JSON « paper bundle » ----------
BUNDLE = {
    "generated_at": STAMP,
    "best_ckpt": str(best_ckpt) if best_ckpt else "n/a",
    "phys_cfg": {
        "zeta": zeta,
        "omega0_rad_s": omega0,
        "fs_hz": fs_hz,
        "time_scale_s": Tphys
    },
    "residual_metrics": {
        "L2_mean": L2_mean,
        "L1_mean": L1_mean,
        "Linf": Linf,
        "abs_h_p95": P95
    },
    "psd": psd_info,
    "figures": FIGS,
    "tables": {"kpis_csv": str(CSV_PATH)},
    "inputs": {
        "diag_npz": str(diag_npz),
        "diag_json": str(diag_json) if diag_json else "n/a",
        "train_history": str(hist4) if hist4 else "n/a"
    }
}
BUNDLE_PATH = ART_DIR / f"paper_bundle_{STAMP}.json"
with open(BUNDLE_PATH, "w") as f:
    json.dump(BUNDLE, f, indent=2)

print("[BUNDLE] paper figures:", FIGS)
print("[BUNDLE] KPIs CSV   ->", CSV_PATH)
print("[BUNDLE] JSON       ->", BUNDLE_PATH)
print("Cellule 7 terminée ✅ — bundle article prêt.")


[BUNDLE] paper figures: {'u': '/home/a.riyahi/spinn_project/outputs/figs/paper_u_20251110_192715.png', 'h': '/home/a.riyahi/spinn_project/outputs/figs/paper_residual_20251110_192715.png', 'hist_h': '/home/a.riyahi/spinn_project/outputs/figs/paper_residual_hist_20251110_192715.png', 'psd': '/home/a.riyahi/spinn_project/outputs/figs/paper_psd_20251110_192715.png', 'train': '/home/a.riyahi/spinn_project/outputs/figs/paper_training_20251110_192715.png'}
[BUNDLE] KPIs CSV   -> /home/a.riyahi/spinn_project/outputs/artifacts/paper_kpis_20251110_192715.csv
[BUNDLE] JSON       -> /home/a.riyahi/spinn_project/outputs/artifacts/paper_bundle_20251110_192715.json
Cellule 7 terminée ✅ — bundle article prêt.


In [10]:
# ============================================================
# Cellule 8 — Évaluation du meilleur modèle (robuste / auto)
#  - Trouve le dernier checkpoint *_best_*.h5 (OUT_DIR/models ou PROJ_DIR/models)
#  - Recharge le modèle (compile auto si possible)
#  - Évalue:
#      * Classification (accuracy, confmat, ROC) si Xte/yte/CLASSES existent et compatibles
#      * Sinon: régression (MSE) minimale sur Xte si dispo
#  - Sauvegarde résultats (JSON) et figures quand pertinent
# ============================================================

from __future__ import annotations
import os, json, glob, datetime
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# --- chemins de base (réutilise ceux définis ailleurs, sinon valeurs par défaut) ---
try:
    PROJ_DIR
except NameError:
    PROJ_DIR = Path("/home/a.riyahi/spinn_project")
try:
    OUT_DIR
except NameError:
    OUT_DIR = PROJ_DIR / "outputs"

MODEL_DIRS = [OUT_DIR / "models", PROJ_DIR / "models"]
for d in [OUT_DIR] + MODEL_DIRS:
    d.mkdir(parents=True, exist_ok=True)

STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# --- utilitaires ---
def _latest_best_ckpt() -> Path | None:
    cands = []
    patterns = ["*best*.h5"]
    for md in MODEL_DIRS:
        for pat in patterns:
            cands += glob.glob(str(md / pat))
    if not cands:
        return None
    cands = sorted(cands, key=lambda p: Path(p).stat().st_mtime, reverse=True)
    return Path(cands[0])

def _safe_name(p: Path | None) -> str:
    return str(p) if p is not None else "n/a"

# --- 1) Localiser et charger le meilleur modèle ---
best_ckpt = _latest_best_ckpt()
if best_ckpt is None or (not best_ckpt.exists()):
    raise FileNotFoundError("Aucun checkpoint '*best*.h5' trouvé dans outputs/models ou models/.")

import tensorflow as tf
print(f"[LOAD] Chargement du modèle: {best_ckpt}")
model = tf.keras.models.load_model(str(best_ckpt), compile=False)

# --- 2) Préparer la loss/metrics en fonction du contexte ---
# Si 'loss_fn' existe déjà dans le notebook, on l’utilise.
loss_to_use = None
try:
    loss_fn  # type: ignore
    loss_to_use = loss_fn
    print("[INFO] loss_fn externe détectée: utilisation directe.")
except NameError:
    pass

# Détection de données test disponibles
have_X = 'Xte' in globals()
have_y = 'yte' in globals()
have_classes = 'CLASSES' in globals()

# Heuristique de loss si non fournie:
if loss_to_use is None:
    if have_y:
        y_arr = np.asarray(yte)
        # Classification si labels entiers et nb_classes>1
        if np.issubdtype(y_arr.dtype, np.integer):
            loss_to_use = 'sparse_categorical_crossentropy'
        else:
            loss_to_use = 'mse'
    else:
        loss_to_use = 'mse'

# Choix métriques
metrics_to_use = []
if loss_to_use == 'sparse_categorical_crossentropy':
    metrics_to_use = ['accuracy']

model.compile(optimizer="adam", loss=loss_to_use, metrics=metrics_to_use)

# --- 3) Évaluation ---
results = {
    "checkpoint": _safe_name(best_ckpt),
    "loss_name": loss_to_use if isinstance(loss_to_use, str) else getattr(loss_to_use, "__name__", "callable"),
    "mode": None,
    "test_loss": None,
    "test_acc": None,
    "roc_auc_macro": None,
}

FIG_DIR = OUT_DIR / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Cas A) Classification supervisée si on a Xte, yte ET une sortie multi-classes
did_classif_eval = False
if have_X and have_y:
    # Tenter classification si loss correspond et la tête du modèle semble multi-classes
    try:
        # petite inférence de shape
        test_logits = model.predict(Xte, verbose=0)
        n_out = int(np.shape(test_logits)[-1]) if np.ndim(test_logits) >= 2 else 1
        is_multiclass = (n_out > 1) and (loss_to_use == 'sparse_categorical_crossentropy')
        if is_multiclass:
            did_classif_eval = True
            print("[EVAL] Mode classification multi-classes détecté.")
            test_loss, test_acc = model.evaluate(Xte, yte, verbose=0)
            results["mode"] = "classification"
            results["test_loss"] = float(test_loss)
            results["test_acc"]  = float(test_acc)
            print(f"[TEST] Loss={test_loss:.4e} | Acc={test_acc:.4f}")

            # --- Prédictions et rapports ---
            y_pred_probs = model.predict(Xte, verbose=0)
            y_pred = np.argmax(y_pred_probs, axis=1)

            # Rapport de classification si classes connues
            if have_classes:
                from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay
                import seaborn as sns

                print("\n[REPORT]")
                print(classification_report(yte, y_pred, target_names=CLASSES))

                # Matrice de confusion
                cm = confusion_matrix(yte, y_pred)
                plt.figure(figsize=(6,5))
                sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                            xticklabels=CLASSES, yticklabels=CLASSES)
                plt.xlabel("Prédit"); plt.ylabel("Réel")
                plt.title("Matrice de Confusion - Test")
                plt.tight_layout()
                cm_path = FIG_DIR / f"confmat_best_{STAMP}.png"
                plt.savefig(cm_path); plt.close()
                print(f"[SAVE] Confusion matrix -> {cm_path}")

                # ROC macro (one-vs-rest)
                try:
                    y_true_oh = tf.keras.utils.to_categorical(yte, num_classes=len(CLASSES))
                    roc_auc_macro = roc_auc_score(y_true_oh, y_pred_probs, multi_class="ovr", average="macro")
                    results["roc_auc_macro"] = float(roc_auc_macro)
                    RocCurveDisplay.from_predictions(y_true_oh.ravel(), y_pred_probs.ravel())
                    plt.title("Courbes ROC (Test)")
                    roc_path = FIG_DIR / f"roc_best_{STAMP}.png"
                    plt.savefig(roc_path); plt.close()
                    print(f"[SAVE] ROC curves -> {roc_path}")
                except Exception as e:
                    print("[WARN] ROC-AUC non calculé:", e)
    except Exception as e:
        print(f"[WARN] Échec tentative classification, fallback régression. Raison: {e}")

# Cas B) Régression / PINN : au minimum, calculer la loss sur Xte si dispo
if (not did_classif_eval) and have_X:
    print("[EVAL] Mode régression / PINN.")
    test_loss = model.evaluate(Xte, yte if have_y else None, verbose=0)
    # Keras peut renvoyer un scalaire ou (loss, metrics...)
    if isinstance(test_loss, (list, tuple)):
        test_loss = test_loss[0]
    results["mode"] = "regression"
    results["test_loss"] = float(test_loss)
    print(f"[TEST] Loss={test_loss:.4e}")

# --- 4) Sauvegarde résultats bruts ---
res_path = OUT_DIR / f"results_test_{STAMP}.json"
with open(res_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[SAVE] Résultats test -> {res_path}")


[LOAD] Chargement du modèle: /home/a.riyahi/spinn_project/models/PINN_2_2_best_20251110_192708.h5
[SAVE] Résultats test -> /home/a.riyahi/spinn_project/outputs/results_test_20251110_192717.json


# --- 9.5 README (Markdown) pour l’article ---
readme_md = f"""# PINN Z24 — Export {RUN_TAG}

**Date :** {STAMP}  
**Best checkpoint :** `{summary['best_ckpt']}`

## 1) Métriques Test (classification)
- Loss : {_fmt(summary['test_metrics'].get('loss'))}
- Accuracy : {_fmt(summary['test_metrics'].get('accuracy'))}
- ROC-AUC (macro) : {_fmt(summary['test_metrics'].get('roc_auc_macro'))}

_Fichier :_ `artifacts/{res_test_path.name if res_test_path else 'results_test_*.json'}`

## 2) Physique (config)
- ζ (zeta) : {_fmt(summary['phys_cfg'].get('zeta'))}
- ω₀ [rad/s] : {_fmt(summary['phys_cfg'].get('omega0_rad_s'))}
- time_scale_s : {_fmt(summary['phys_cfg'].get('time_scale_s'))}
- fs_hz : {_fmt(summary['phys_cfg'].get('fs_hz'))}

_Fichier :_ `artifacts/{phys_cfg_path.name if phys_cfg_path else 'pinn_phys_config_*.json'}`

## 3) Diagnostics PINN
- L2_mean(h) : {_fmt((summary.get('diagnostics') or {}).get('L2_mean'))}
- L1_mean(h) : {_fmt((summary.get('diagnostics') or {}).get('L1_mean'))}
- Linf(h)    : {_fmt((summary.get('diagnostics') or {}).get('Linf'))}
- abs_h_p95  : {_fmt((summary.get('diagnostics') or {}).get('abs_h_p95'))}

_Fichiers :_ `artifacts/{(diag_json_path.name if diag_json_path else 'diagnostics_summary*.json')}`,  
`artifacts/{(diag_npz_path.name if diag_npz_path else 'pinn*_diagnostics_*.npz')}`

## 4) Jeux de données
- Manifest : `artifacts/{manifest_path.name if manifest_path else 'manifest_*.json'}`
- Cache dataset : `artifacts/{cache_path.name if cache_path else 'dataset_cache.npz'}`

## 5) Figures incluses
- Matrice de confusion, ROC, courbes de pertes, u(t), PSD(u), résidu h(t)
- Dossier : `figs/` (jusqu’à 20 figures récentes pertinentes)

## 6) Reproduire / Évaluer
1. Créer un venv et installer les dépendances requises (TensorFlow, numpy, scipy, scikit-learn, matplotlib, seaborn, h5py, etc.).
2. Charger le modèle :
   ```python
   import tensorflow as tf
   model = tf.keras.models.load_model("models/{best_ckpt_name if best_ckpt_name else 'BEST.h5'}", compile=False)


In [11]:
# ============================================================
# Cellule 10 — Packaging "paper-ready" (V2.3)
# - Agrège manifestes, historiques et diagnostics récents
# - Produit des tables CSV prêtes à être importées (article)
# - Recopie les figures clés dans un dossier unique
# - Écrit un manifeste JSON récapitulatif
# ============================================================

from __future__ import annotations
import os, re, json, glob, shutil, time, math, datetime as dt
from pathlib import Path
from typing import List, Dict, Any
import numpy as np

# --- 10.0 Chemins / I/O -------------------------------------------------------
PROJ_DIR   = Path(os.environ.get("SPINN_HOME", "/home/a.riyahi/spinn_project"))
OUT_DIR    = PROJ_DIR / "outputs"
ART_DIR    = OUT_DIR / "artifacts"
FIG_DIR    = OUT_DIR / "figs"
MODEL_DIR  = PROJ_DIR / "models"
PAPER_DIR  = OUT_DIR / "paper_v2_3"
PAPER_FIGS = PAPER_DIR / "figs"
PAPER_TAB  = PAPER_DIR / "tables"
for d in [PAPER_DIR, PAPER_FIGS, PAPER_TAB]:
    d.mkdir(parents=True, exist_ok=True)

STAMP = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

def _latest(patterns: List[str], root: Path) -> Path|None:
    cands = []
    for pat in patterns:
        cands += list(root.glob(pat))
    if not cands:
        return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]

def _read_json(p: Path) -> dict:
    try:
        with open(p, "r") as f:
            return json.load(f)
    except Exception:
        return {}

def _safe_float(x, default=float("nan")) -> float:
    try:
        return float(x)
    except Exception:
        return default

# --- 10.1 Récup artefacts clés -------------------------------------------------
# Manifest dataset (V2.3 si présent, sinon V2.2)
manifest_paths = sorted(ART_DIR.glob("manifest_pinn2_3*.json")) or sorted(ART_DIR.glob("manifest_pinn2_2*.json"))
manifest_ds = _read_json(manifest_paths[-1]) if manifest_paths else {}

# Config physique
phys_cfg_path = ART_DIR / "pinn_phys_config_v2_3.json"
if not phys_cfg_path.exists():
    phys_cfg_path = ART_DIR / "pinn_phys_config_v2_2.json"
PHYS_CFG = _read_json(phys_cfg_path)

# Historique d’entraînement (cellule 4/7/8/9 — on prend le + récent "train_history_*.json")
hist_paths = sorted(ART_DIR.glob("train_history_*.json"), key=lambda p: p.stat().st_mtime)
hist_last  = _read_json(hist_paths[-1]) if hist_paths else {}

# Diagnostics (le + récent "diagnostics_summary*.json")
diag_paths = sorted(ART_DIR.glob("diagnostics_summary*.json"), key=lambda p: p.stat().st_mtime)
diag_last  = _read_json(diag_paths[-1]) if diag_paths else {}

# Meilleur checkpoint (préférence V2.3 → V2.2)
best_ckpt = _latest(["PINN_2_3_best_*.h5", "PINN_2_2_best_*.h5", "pinn_u_only_best_*.h5"], MODEL_DIR)
best_ckpt = str(best_ckpt) if best_ckpt else "n/a"

# --- 10.2 Tables CSV -----------------------------------------------------------
def write_csv(path: Path, header: List[str], rows: List[List[Any]]):
    import csv
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        for r in rows:
            w.writerow(r)

# (A) Table dataset (classes / comptes)
rows_ds = []
if manifest_ds:
    classes = manifest_ds.get("classes") or list((manifest_ds.get("per_class") or {}).keys())
    per_class = manifest_ds.get("per_class") or manifest_ds.get("counts") or {}
    total_mat = manifest_ds.get("total_mat") or sum(per_class.values()) if per_class else ""
    for c in classes or []:
        rows_ds.append([c, per_class.get(c, "")])
    rows_ds.append(["TOTAL", total_mat])

write_csv(PAPER_TAB / "table_dataset_counts.csv", ["Classe", "Nombre_de_.mat"], rows_ds)

# (B) Table hyperparamètres / physique
rows_phys = [
    ["zeta",             PHYS_CFG.get("zeta", "")],
    ["omega0_rad_s",     PHYS_CFG.get("omega0_rad_s", "")],
    ["fs_hz",            PHYS_CFG.get("fs_hz", "")],
    ["L",                PHYS_CFG.get("L", "")],
    ["time_scale_s",     PHYS_CFG.get("time_scale_s", "")],
    ["seed",             PHYS_CFG.get("seed", "")],
    ["n_coll",           PHYS_CFG.get("n_coll", PHYS_CFG.get("n_collocation", ""))],
    ["mlp_widths",       PHYS_CFG.get("mlp_widths", [64,64,64])],
    ["activation",       PHYS_CFG.get("activation", "tanh")],
]
write_csv(PAPER_TAB / "table_phys_hparams.csv", ["Paramètre", "Valeur"], rows_phys)

# (C) Table entraînement (dernière session)
epochs  = hist_last.get("epoch")
if isinstance(epochs, list) and epochs:
    n_epochs = int(epochs[-1])
else:
    # autres historiques (clé "history" → liste de dicts)
    history = hist_last.get("history", [])
    n_epochs = int(history[-1].get("epoch", len(history))) if history else 0

# On essaie d’extraire les pertes finales si disponibles
def _last(name: str, dflt=np.nan):
    v = hist_last.get(name)
    if isinstance(v, list) and v:
        return _safe_float(v[-1], dflt)
    return dflt

row_train = [
    ["epochs_run", n_epochs],
    ["L_total_last", _last("L_tot")],
    ["L_phys_last",  _last("L_phys")],
    ["L_data_last",  _last("L_data")],
    ["lr_last",      _last("lr")],
    ["best_ckpt",    best_ckpt],
]
write_csv(PAPER_TAB / "table_training_summary.csv", ["Clé", "Valeur"], row_train)

# (D) Table diagnostics résidu
diag_metrics = (diag_last.get("metrics") or diag_last.get("summary_residuals") or {})
rows_diag = [
    ["L2_mean",    diag_metrics.get("L2_mean", "")],
    ["L1_mean",    diag_metrics.get("L1_mean", "")],
    ["Linf",       diag_metrics.get("Linf", "")],
    ["abs_h_p95",  diag_metrics.get("abs_h_p95", "")],
    ["zeta_used",  diag_metrics.get("zeta", PHYS_CFG.get("zeta", ""))],
    ["omega0_used",diag_metrics.get("omega0", PHYS_CFG.get("omega0_rad_s", ""))],
    ["time_scale_s_used", diag_metrics.get("time_scale_s", PHYS_CFG.get("time_scale_s", ""))],
]
write_csv(PAPER_TAB / "table_diagnostics_residu.csv", ["Métrique", "Valeur"], rows_diag)

print(f"[SAVE] tables -> {PAPER_TAB}")

# --- 10.3 Figures : copie des plus récentes -----------------------------------
# On prend la dernière occurrence pour chaque famille de figures
def copy_latest(pattern: str, dst_name: str) -> str|None:
    files = sorted(FIG_DIR.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not files:
        return None
    src = files[-1]
    dst = PAPER_FIGS / dst_name
    shutil.copy2(src, dst)
    return str(dst)

fig_map = {
    "pinn_u_*.png":        "Fig_u.png",
    "pinn_udot_*.png":     "Fig_udot.png",
    "pinn_uddot_*.png":    "Fig_uddot.png",
    "pinn_residual_*.png": "Fig_residu.png",
    "pinn_psd_*.png":      "Fig_psd.png",
    "pinn_cell4_losses_*.png": "Fig_losses_cell4.png",
}

copied = {}
for pat, name in fig_map.items():
    path = copy_latest(pat, name)
    if path:
        copied[pat] = path

print(f"[SAVE] figs -> {PAPER_FIGS}")

# --- 10.4 Manifeste paper ------------------------------------------------------
paper_manifest = {
    "created_at": STAMP,
    "best_ckpt": best_ckpt,
    "phys_cfg_path": str(phys_cfg_path),
    "dataset_manifest": str(manifest_paths[-1]) if manifest_paths else "n/a",
    "train_history": str(hist_paths[-1]) if hist_paths else "n/a",
    "diagnostics_summary": str(diag_paths[-1]) if diag_paths else "n/a",
    "tables": {
        "dataset_counts": str(PAPER_TAB / "table_dataset_counts.csv"),
        "phys_hparams":   str(PAPER_TAB / "table_phys_hparams.csv"),
        "training":       str(PAPER_TAB / "table_training_summary.csv"),
        "diagnostics":    str(PAPER_TAB / "table_diagnostics_residu.csv"),
    },
    "figures": copied,
    "notes": "Dossier prêt pour l’article (figures + tables).",
}
with open(PAPER_DIR / f"paper_manifest_{STAMP}.json", "w") as f:
    json.dump(paper_manifest, f, indent=2)

print(f"[SAVE] paper_manifest -> {PAPER_DIR / f'paper_manifest_{STAMP}.json'}")
print("Cellule 10 terminée ✅ — assets 'paper-ready' générés.")


[SAVE] tables -> /home/a.riyahi/spinn_project/outputs/paper_v2_3/tables
[SAVE] figs -> /home/a.riyahi/spinn_project/outputs/paper_v2_3/figs
[SAVE] paper_manifest -> /home/a.riyahi/spinn_project/outputs/paper_v2_3/paper_manifest_20251110_192717.json
Cellule 10 terminée ✅ — assets 'paper-ready' générés.


In [12]:
# --- 11.0 • Détection rapide du type de modèle (PINN vs NN classif/régression) ---
import tensorflow as tf
import numpy as np
from pathlib import Path
import json, math, os

# On suppose que 'model' ou 'pinn' existe déjà (sinon on tente de charger le best ckpt)
def _latest_best_ckpt(model_dirs):
    cands = []
    for d in model_dirs:
        if not Path(d).exists(): 
            continue
        cands += list(Path(d).glob("*best*.h5"))
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return str(cands[0]) if cands else None

PROJECT_ROOT = Path("/home/a.riyahi/spinn_project")
OUT_DIR  = PROJECT_ROOT / "outputs"
ART_DIR  = OUT_DIR / "artifacts"
CACHE_DIR= OUT_DIR / "cache"
MODEL_DIRS = [str(PROJECT_ROOT / "models"), str(OUT_DIR / "models")]

if 'model' in globals() and isinstance(model, tf.keras.Model):
    _mdl = model
elif 'pinn' in globals() and isinstance(pinn, tf.keras.Model):
    _mdl = pinn
else:
    ckpt = _latest_best_ckpt(MODEL_DIRS)
    assert ckpt is not None, "Aucun checkpoint '*best*.h5' trouvé."
    _mdl = tf.keras.models.load_model(ckpt, compile=False)
    print(f"[LOAD] Modèle chargé: {ckpt}")

inp_shape = _mdl.inputs[0].shape  # ex: (None, 1) pour PINN
is_pinn = (int(inp_shape[-1]) == 1 and len(inp_shape) == 2)
print(f"[CHECK] input_shape={inp_shape} → is_pinn={is_pinn}")


[CHECK] input_shape=(None, 1) → is_pinn=True


In [13]:
# --- 11.1 • Évaluation PINN sur t_norm (pas de X_test !) ---
import numpy as np
import tensorflow as tf
import json, math, datetime
from pathlib import Path

assert is_pinn, "Ce bloc est pour PINN. (input shape (None,1))."

# Charger L depuis le cache si possible
DEFAULT_L = 65536
try:
    ds = np.load(CACHE_DIR / "pinn2_2_dataset_cache.npz", allow_pickle=False)
    L = int(ds["X_train"].shape[1])
except Exception:
    L = DEFAULT_L

# Charger config physique si dispo (v2.3 puis v2.2)
phys_cfg_path = None
for cand in [ART_DIR / "pinn_phys_config_v2_3.json", ART_DIR / "pinn_phys_config_v2_2.json"]:
    if cand.exists():
        phys_cfg_path = cand
        break

PHYS_CFG = {"zeta":0.01, "omega0_rad_s":2*math.pi*4.0, "time_scale_s":0.0, "fs_hz":100.0}
if phys_cfg_path:
    try:
        PHYS_CFG.update(json.loads(Path(phys_cfg_path).read_text()))
    except Exception:
        pass

zeta   = float(PHYS_CFG.get("zeta", 0.01))
omega0 = float(PHYS_CFG.get("omega0_rad_s", 2*math.pi*4.0))
Tphys  = float(PHYS_CFG.get("time_scale_s", 0.0)) or 0.0
fs_hz  = float(PHYS_CFG.get("fs_hz", 100.0))

# Grille de temps normalisée
t_norm = tf.linspace(0.0, 1.0, L)[:, None]  # (L,1)

# Prédiction u(t) + dérivées via autograd
with tf.GradientTape(persistent=True) as g2:
    g2.watch(t_norm)
    with tf.GradientTape() as g1:
        g1.watch(t_norm)
        u = _mdl(t_norm, training=False)   # (L,1)
    u_dot = g1.gradient(u, t_norm)         # du/dt_norm
u_ddot = g2.gradient(u_dot, t_norm)        # d2u/dt_norm2
del g1, g2

# Rescale temporel si Tphys>0
if Tphys > 0:
    invT  = tf.constant(1.0/Tphys, tf.float32)
    u_dot  = u_dot  * invT
    u_ddot = u_ddot * (invT*invT)

# Résidu physique: h = ü + 2ζω0 u̇ + ω0² u
z = tf.constant(zeta,  tf.float32)
w = tf.constant(omega0,tf.float32)
h = u_ddot + 2.0*z*w*u_dot + (w*w)*u

# Sauvegarde NPZ compacte pour réutilisation
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
npz_path = ART_DIR / f"pinn_diag_{stamp}.npz"
np.savez_compressed(
    npz_path,
    t_norm=t_norm.numpy().astype(np.float32).ravel(),
    u=u.numpy().astype(np.float32).ravel(),
    u_dot=u_dot.numpy().astype(np.float32).ravel(),
    u_ddot=u_ddot.numpy().astype(np.float32).ravel(),
    h=h.numpy().astype(np.float32).ravel(),
    zeta=np.float32(zeta), omega0=np.float32(omega0),
    time_scale_s=np.float32(Tphys), fs_hz=np.float32(fs_hz)
)
print(f"[SAVE] diagnostics npz -> {npz_path}")


[SAVE] diagnostics npz -> /home/a.riyahi/spinn_project/outputs/artifacts/pinn_diag_20251110_192717.npz


In [14]:
# --- 11.2 • KPI résidu + PSD et exports article ---
import numpy as np, json, datetime
from pathlib import Path

# Recharger le dernier diag si on relance la cellule seule
latest = sorted(ART_DIR.glob("pinn_diag_*.npz"), key=lambda p: p.stat().st_mtime)
assert latest, "Aucun fichier pinn_diag_*.npz trouvé. Lance 11.1 d'abord."
D = np.load(latest[-1], allow_pickle=False)

t_norm = D["t_norm"]
u      = D["u"]
h      = D["h"]
fs_hz  = float(D.get("fs_hz", 0.0))
Tphys  = float(D.get("time_scale_s", 0.0))

# KPI résidu
L2_mean = float(np.mean(h**2))
L1_mean = float(np.mean(np.abs(h)))
Linf    = float(np.max(np.abs(h)))
P95     = float(np.percentile(np.abs(h), 95.0))

# PSD (Welch si scipy dispo, sinon rFFT bins)
omega_est = None
try:
    from scipy.signal import welch, find_peaks
    if fs_hz > 0:
        f, Puu = welch(u, fs=fs_hz, nperseg=min(8192, len(u)))
        pk, _ = find_peaks(Puu)
        fpk = float(f[pk[np.argmax(Puu[pk])]]) if pk.size>0 else float(f[np.argmax(Puu)])
        omega_est = float(2*np.pi*fpk)
    else:
        # bins t_norm
        U = np.fft.rfft(u - u.mean())
        P = (np.abs(U)**2)/max(1,len(U))
        fbins = np.fft.rfftfreq(len(u), d=1.0/len(u))
        fpk = float(fbins[np.argmax(P)])
        omega_est = float(2*np.pi*fpk)
except Exception:
    pass

summary = {
    "L2_mean": L2_mean, "L1_mean": L1_mean, "Linf": Linf, "abs_h_p95": P95,
    "omega_est_rad_s": omega_est, "fs_hz": fs_hz, "time_scale_s": Tphys
}

# Exports
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = ART_DIR / f"pinn_kpis_{stamp}.json"
csv_path  = ART_DIR / f"pinn_kpis_{stamp}.csv"

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

with open(csv_path, "w") as f:
    f.write("metric,value\n")
    for k,v in summary.items():
        f.write(f"{k},{'' if v is None else v}\n")

print(f"[SAVE] KPIs JSON -> {json_path}")
print(f"[SAVE] KPIs CSV  -> {csv_path}")


[SAVE] KPIs JSON -> /home/a.riyahi/spinn_project/outputs/artifacts/pinn_kpis_20251110_192717.json
[SAVE] KPIs CSV  -> /home/a.riyahi/spinn_project/outputs/artifacts/pinn_kpis_20251110_192717.csv


In [15]:
# === Cell 11.2-bis — omega_est robuste (bande limitée 0.5–15 Hz) ===
import numpy as np, json
from pathlib import Path
from scipy.signal import welch, find_peaks

latest = sorted(ART_DIR.glob("pinn_diag_*.npz"), key=lambda p: p.stat().st_mtime)
assert latest, "Lance 11.1/11.2 d'abord."
D = np.load(latest[-1], allow_pickle=False)
u     = D["u"].astype(float).ravel()
fs_hz = float(D.get("fs_hz", 0.0))

# Bande d'intérêt (à ajuster si besoin)
FMIN, FMAX = 0.5, 15.0

if fs_hz <= 0:
    # fallback sans fs : on normalise en bins ~Hz via longueur
    fs_used = len(u)
else:
    fs_used = fs_hz

# Detrend léger
u = u - u.mean()

# Welch + masquage bande
f, Pxx = welch(u, fs=fs_used, nperseg=min(8192, len(u)))
mask = (f >= FMIN) & (f <= FMAX)
f_sel, P_sel = f[mask], Pxx[mask]

if f_sel.size == 0:
    # au cas où, fallback global
    fpk = float(f[np.argmax(Pxx)])
else:
    peaks, _ = find_peaks(P_sel)
    fpk = float(f_sel[peaks[np.argmax(P_sel[peaks])]]) if peaks.size else float(f_sel[np.argmax(P_sel)])

omega_est = 2*np.pi*fpk

# Mise à jour du dernier JSON KPI (le plus récent)
kpi_list = sorted(ART_DIR.glob("pinn_kpis_*.json"), key=lambda p: p.stat().st_mtime)
assert kpi_list, "pinn_kpis_*.json introuvable (lance 11.2)."
kpi_path = kpi_list[-1]
kpi = json.loads(kpi_path.read_text())
kpi["omega_est_rad_s"] = float(omega_est)
kpi_path.write_text(json.dumps(kpi, indent=2))
print("[UPDATE] omega_est_rad_s =", omega_est, "->", kpi_path)

# Regénère la figure PSD ciblée (optionnel)
import matplotlib.pyplot as plt
plt.figure(figsize=(6.0,3.0))
plt.semilogy(f, Pxx + 1e-20, label="PSD (Welch)")
plt.axvspan(FMIN, FMAX, color="orange", alpha=0.15, label=f"Bande [{FMIN},{FMAX}] Hz")
plt.axvline(fpk, color="r", linewidth=1.5, label=f"f_peak≈{fpk:.3f} Hz")
plt.xlabel("Frequency [Hz]"); plt.ylabel("PSD(u)"); plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR/"pinn_psd_u_article.png", dpi=300); plt.close()
print("[SAVE] PSD ciblée ->", FIG_DIR/"pinn_psd_u_article.png")


[UPDATE] omega_est_rad_s = 43.1815591789808 -> /home/a.riyahi/spinn_project/outputs/artifacts/pinn_kpis_20251110_192717.json
[SAVE] PSD ciblée -> /home/a.riyahi/spinn_project/outputs/figs/pinn_psd_u_article.png


In [16]:
# --- 11.3 • Figures prêtes article (u, résidu, PSD) ---
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

D = np.load(sorted(ART_DIR.glob("pinn_diag_*.npz"))[-1], allow_pickle=False)
t_norm = D["t_norm"]; u = D["u"]; h = D["h"]
fs_hz  = float(D.get("fs_hz", 0.0)); Tphys=float(D.get("time_scale_s", 0.0))

FIG_DIR = OUT_DIR / "figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
x = (t_norm * Tphys) if Tphys>0 else t_norm
xlab = "t [s]" if Tphys>0 else "t_norm"

def _save(fig, name):
    path = FIG_DIR / name
    fig.tight_layout(); fig.savefig(path, dpi=300); plt.close(fig)
    print(f"[SAVE] {path}")

# (a) u(t) — zoom fenêtre initiale si très long
nwin = min(5000, len(u))
fig = plt.figure(figsize=(6.0, 2.6))
plt.plot(x[:nwin], u[:nwin], linewidth=1.5)
plt.xlabel(xlab); plt.ylabel("u"); plt.grid(True, alpha=.3)
_save(fig, "pinn_u_article.png")

# (b) h(t)
fig = plt.figure(figsize=(6.0, 2.6))
plt.plot(x[:nwin], h[:nwin], linewidth=1.3)
plt.xlabel(xlab); plt.ylabel("h"); plt.grid(True, alpha=.3)
_save(fig, "pinn_residual_article.png")

# (c) PSD (u)
try:
    from scipy.signal import welch
    if fs_hz > 0:
        f, Puu = welch(u, fs=fs_hz, nperseg=min(8192, len(u)))
        xlab_psd = "Frequency [Hz]"
    else:
        f, Puu = welch(u, fs=len(u), nperseg=min(8192, len(u)))
        xlab_psd = "Frequency (bins)"
    fig = plt.figure(figsize=(6.0, 3.0))
    plt.semilogy(f, Puu + 1e-18)
    plt.xlabel(xlab_psd); plt.ylabel("PSD"); plt.grid(True, which="both", alpha=.3)
    _save(fig, "pinn_psd_u_article.png")
except Exception:
    pass


[SAVE] /home/a.riyahi/spinn_project/outputs/figs/pinn_u_article.png
[SAVE] /home/a.riyahi/spinn_project/outputs/figs/pinn_residual_article.png
[SAVE] /home/a.riyahi/spinn_project/outputs/figs/pinn_psd_u_article.png


In [17]:
# --- 11.4 • Ligne récap pour Table 1 (PINN) ---
import json, csv
from pathlib import Path
import numpy as np

latest_json = sorted(ART_DIR.glob("pinn_kpis_*.json"), key=lambda p: p.stat().st_mtime)
assert latest_json, "Lance 11.2 d'abord."
kpis = json.loads(latest_json[-1].read_text())

# Exemple de ligne pour fusionner ensuite avec NN V1/V2/V3 (accuracy/AUC) côté Word
row = {
    "Model": "PINN V1",
    "Validation_Accuracy": "",      # non pertinent pour PINN ici
    "AUC_macro": "",
    "Residual_L2_mean": f"{kpis['L2_mean']:.3e}",
    "Residual_Linf":   f"{kpis['Linf']:.3e}",
    "Residual_p95":    f"{kpis['abs_h_p95']:.3e}",
    "Omega_est_rad_s": ("" if kpis.get("omega_est_rad_s") is None else f"{kpis['omega_est_rad_s']:.4f}")
}

csv_out = ART_DIR / "table1_pinn_row.csv"
with open(csv_out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(row.keys()))
    w.writeheader(); w.writerow(row)

print(f"[SAVE] → {csv_out}")
print(row)


[SAVE] → /home/a.riyahi/spinn_project/outputs/artifacts/table1_pinn_row.csv
{'Model': 'PINN V1', 'Validation_Accuracy': '', 'AUC_macro': '', 'Residual_L2_mean': '1.205e-03', 'Residual_Linf': '7.372e-02', 'Residual_p95': '6.660e-02', 'Omega_est_rad_s': '43.1816'}


In [3]:
# 7.2a — Créer la ligne PINN (CSV) à partir des valeurs connues
import pandas as pd
from pathlib import Path

PACK = Path("reviewer_pack"); PACK.mkdir(parents=True, exist_ok=True)

row = {
    "Model": "PINN V1",
    "Val_Accuracy_(%)": None,     # N/A pour PINN si régression/physique
    "Macro_F1_(%)":     None,
    "ROC_AUC_macro":    None,
    "False_Negatives":  None,
    # ← valeurs que tu as déjà partagées
    "PINN_L2_mean(h)":  1.205e-03,
    "PINN_abs_h_p95":   6.660e-02,
    "omega0_rad_s":     25.132742,
    "omega_est_rad_s":  43.181559,
    "abs_rel_err_omega(%)": 71.81,
    "Notes": "physics residual; SDOF consistency",
}
df = pd.DataFrame([row])
out_csv = PACK / "table1_pinn_row.csv"
df.to_csv(out_csv, index=False)
print("[OK] Écrit :", out_csv.resolve())
df


[OK] Écrit : /home/a.riyahi/spinn_project/reviewer_pack/table1_pinn_row.csv


,Model,Val_Accuracy_(%),Macro_F1_(%),ROC_AUC_macro,False_Negatives,PINN_L2_mean(h),PINN_abs_h_p95,omega0_rad_s,omega_est_rad_s,abs_rel_err_omega(%),Notes
0,PINN V1,None,None,None,None,0.001205,0.0666,25.132742,43.181559,71.81,physics residual; SDOF consistency


In [4]:
# === 11.6 — Fusion Table 1 (Final+) ===
import json, re
import numpy as np
import pandas as pd
from pathlib import Path

# ---- Racines cohérentes avec tes notebooks ----
try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path("/home/a.riyahi/spinn_project")

OUT_DIR = PROJECT_ROOT / "outputs"
ART_DIR = OUT_DIR / "artifacts"     # pinn_kpis_*.json, pinn_diag_*.npz
TAB_DIR = OUT_DIR / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

NN_DIR  = PROJECT_ROOT / "NN"       # Comparisons + reviewer_pack vivent ici

def newest_under(base: Path, patterns):
    hits = []
    for pat in patterns:
        hits.extend(base.rglob(pat))
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

def all_under(base: Path, patterns):
    hits = []
    for pat in patterns:
        hits.extend(base.rglob(pat))
    return sorted(hits, key=lambda p: p.stat().st_mtime)

def norm_model(s: str) -> str:
    s = (s or "").strip().lower().replace("_"," ")
    s = re.sub(r"\s+", " ", s)
    if re.search(r"\b(nn\s*v?1|nn1|v1)\b", s): return "NN V1"
    if re.search(r"\b(nn\s*v?2|nn2|v2)\b", s): return "NN V2"
    if re.search(r"\b(nn\s*v?3|nn3|v3)\b", s): return "NN V3"
    return s.upper()

def to_percent(x):
    if x in ("", None) or (isinstance(x, float) and np.isnan(x)): return ""
    try:
        v = float(x)
        return round(100.0*v, 2) if v <= 1.0 else round(v, 2)
    except Exception:
        return x

def auc_fmt(x):
    if x in ("", None) or (isinstance(x, float) and np.isnan(x)): return ""
    try:
        return round(float(x), 3)
    except Exception:
        return x

def sci(x):
    if x in ("", None) or (isinstance(x, float) and np.isnan(x)): return ""
    try:
        return f"{float(x):.3e}"
    except Exception:
        return x

# ---------- 1) Ligne PINN depuis KPI/NPZ ----------
kpi_json = newest_under(ART_DIR, ["pinn_kpis_*.json"])
if not kpi_json:
    raise RuntimeError("Aucun KPI PINN dans outputs/artifacts (exécute 11.2 puis 11.2-bis).")
kpi = json.loads(kpi_json.read_text())

diag_npz = newest_under(ART_DIR, ["pinn_diag_*.npz"])
omega0 = ""
if diag_npz:
    D = np.load(diag_npz, allow_pickle=False)
    if "omega0" in D.files:
        omega0 = float(D["omega0"])

omega_est = kpi.get("omega_est_rad_s", "")
abs_rel = ""
try:
    if omega0 not in ("", None) and omega_est not in ("", None):
        abs_rel = 100.0 * abs(float(omega_est) - float(omega0)) / float(omega0)
except Exception:
    abs_rel = ""

row_pinn = {
    "Model":"PINN V1",
    "Val_Accuracy_(%)":"",
    "Macro_F1_(%)":"",
    "ROC_AUC_macro":"",
    "False_Negatives":"",
    "PINN_L2_mean(h)": kpi.get("L2_mean",""),
    "PINN_abs_h_p95": kpi.get("abs_h_p95",""),
    "omega0_rad_s": omega0,
    "omega_est_rad_s": omega_est if omega_est not in ("", None) else "",
    "abs_rel_err_omega(%)": abs_rel,
    "Notes":"physics residual; SDOF consistency"
}

# ---------- 2) Lire toutes les métriques NN disponibles sous NN/ ----------
mc_files  = all_under(NN_DIR, ["metrics_comparison*.csv"])
mpc_files = all_under(NN_DIR, ["metrics_per_class*.csv"])
fn_files  = all_under(NN_DIR, ["false_negatives*.csv"])

acc_map, f1_map, auc_map, fn_map = {}, {}, {}, {}
models_seen = set()

# 2.a) Agréger tous les metrics_comparison*.csv (on prend la dernière valeur pour un modèle)
for mc_path in mc_files:
    try:
        mc = pd.read_csv(mc_path)
        mc.columns = [c.strip().lower() for c in mc.columns]
        c_m = "model" if "model" in mc.columns else None
        c_acc = next((c for c in ["val_accuracy","accuracy","acc"] if c in mc.columns), None)
        c_f1  = next((c for c in ["macro_f1","f1_macro","macrof1","f1"] if c in mc.columns), None)
        c_auc = next((c for c in ["macro_auc","roc_auc_macro","auc_macro","auc"] if c in mc.columns), None)
        if not c_m: 
            continue
        for _, r in mc.iterrows():
            mdl = norm_model(str(r[c_m])); models_seen.add(mdl)
            if c_acc is not None: acc_map[mdl] = to_percent(r[c_acc])
            if c_f1  is not None: f1_map[mdl]  = to_percent(r[c_f1])
            if c_auc is not None:
                try: auc_map[mdl] = float(r[c_auc])
                except Exception: pass
    except Exception:
        pass

# 2.b) False negatives
for fn_path in fn_files:
    try:
        fn = pd.read_csv(fn_path)
        fn.columns = [c.strip().lower() for c in fn.columns]
        cm = "model" if "model" in fn.columns else None
        cf = next((c for c in ["false_negatives","fn","falsenegatives"] if c in fn.columns), None)
        if cm and cf:
            for _, r in fn.iterrows():
                mdl = norm_model(str(r[cm])); models_seen.add(mdl)
                try: fn_map[mdl] = float(r[cf])
                except Exception: fn_map[mdl] = r[cf]
    except Exception:
        pass

# 2.c) Si AUC manquant, moyenne depuis metrics_per_class*.csv
if any(mdl not in auc_map for mdl in ["NN V1","NN V2","NN V3"]):
    for mpc_path in mpc_files:
        try:
            mpc = pd.read_csv(mpc_path)
            mpc.columns = [c.strip().lower() for c in mpc.columns]
            if "model" not in mpc.columns:
                continue
            auc_col = next((c for c in mpc.columns if "auc" in c), None)
            if not auc_col:
                continue
            mpc["_model_norm"] = mpc["model"].astype(str).apply(norm_model)
            grp = mpc.groupby("_model_norm")[auc_col].mean()
            for k, v in grp.items():
                try:
                    if k not in auc_map: auc_map[k] = float(v)
                except Exception:
                    pass
        except Exception:
            pass

# ---------- 3) Construire la table ----------
order = ["NN V1","NN V2","NN V3"]
rows = []
for mdl in order:
    rows.append([
        mdl,
        acc_map.get(mdl, ""),
        f1_map.get(mdl,  ""),
        auc_fmt(auc_map.get(mdl, "")),
        fn_map.get(mdl, ""),
        "", "", "", "", "",
        "data-driven"
    ])

rows.append([
    row_pinn["Model"],
    row_pinn["Val_Accuracy_(%)"],
    row_pinn["Macro_F1_(%)"],
    row_pinn["ROC_AUC_macro"],
    row_pinn["False_Negatives"],
    sci(row_pinn["PINN_L2_mean(h)"]),
    sci(row_pinn["PINN_abs_h_p95"]),
    row_pinn["omega0_rad_s"],
    row_pinn["omega_est_rad_s"],
    round(row_pinn["abs_rel_err_omega(%)"], 2) if row_pinn["abs_rel_err_omega(%)"] not in ("", None) else "",
    row_pinn["Notes"]
])

cols = ["Model","Val_Accuracy_(%)","Macro_F1_(%)","ROC_AUC_macro","False_Negatives",
        "PINN_L2_mean(h)","PINN_abs_h_p95","omega0_rad_s","omega_est_rad_s","abs_rel_err_omega(%)","Notes"]
table1 = pd.DataFrame(rows, columns=cols)

# ---------- 4) DIAGNOSTIC + surcharge manuelle éventuelle ----------
missing_auc = [m for m in order if table1.loc[table1["Model"].eq(m),"ROC_AUC_macro"].iloc[0] in ("", None, np.nan)]
if missing_auc:
    print("[WARN] AUC manquante pour:", missing_auc)
    print("[HINT] Modèles détectés dans les fichiers:", sorted(models_seen))
    # >>>> Active la surcharge manuelle si tu connais les AUC (0–1) :
    MANUAL_AUC = {}  # ex: {"NN V1":0.952, "NN V2":0.946, "NN V3":0.958}
    for mdl, val in MANUAL_AUC.items():
        table1.loc[table1["Model"].eq(mdl), "ROC_AUC_macro"] = auc_fmt(val)

# ---------- 5) Sauvegarde ----------
out_csv = TAB_DIR / "table1_summary.csv"
table1.to_csv(out_csv, index=False)
display(table1)

print("\n[INFO] Ecrits dans:", out_csv)
print("[INFO] Sources NN — metrics_comparison:", [p.name for p in mc_files][-3:])
print("[INFO] Sources NN — metrics_per_class:", [p.name for p in mpc_files][-3:])
print("[INFO] Sources NN — false_negatives  :", [p.name for p in fn_files][-3:])
print("[INFO] Sources PINN — KPI:", kpi_json.name, "| DIAG:", diag_npz.name if diag_npz else "n/a")


[WARN] AUC manquante pour: ['NN V1', 'NN V2', 'NN V3']
[HINT] Modèles détectés dans les fichiers: ['NN V1', 'NN V2', 'NN V3']


,Model,Val_Accuracy_(%),Macro_F1_(%),ROC_AUC_macro,False_Negatives,PINN_L2_mean(h),PINN_abs_h_p95,omega0_rad_s,omega_est_rad_s,abs_rel_err_omega(%),Notes
0,NN V1,97.71,97.76,,0.0,,,,,,data-driven
1,NN V2,97.25,97.14,,0.0,,,,,,data-driven
2,NN V3,97.71,97.6,,0.0,,,,,,data-driven
3,PINN V1,,,,,1.205e-03,6.660e-02,25.132742,43.181559,71.81,physics residual; SDOF consistency



[INFO] Ecrits dans: /home/a.riyahi/spinn_project/outputs/tables/table1_summary.csv
[INFO] Sources NN — metrics_comparison: ['metrics_comparison.csv', 'metrics_comparison.csv', 'metrics_comparison_rounded.csv']
[INFO] Sources NN — metrics_per_class: ['metrics_per_class.csv', 'metrics_per_class.csv', 'metrics_per_class_rounded.csv']
[INFO] Sources NN — false_negatives  : ['false_negatives.csv']
[INFO] Sources PINN — KPI: pinn_kpis_20251110_192717.json | DIAG: pinn_diag_20251110_192717.npz


In [5]:
[WARN] AUC manquante pour: ['NN V1', 'NN V2', 'NN V3']
[HINT] Modèles détectés dans les fichiers: ['NN V1', 'NN V2', 'NN V3']

	Model 	Val_Accuracy_(%) 	Macro_F1_(%) 	ROC_AUC_macro 	False_Negatives 	PINN_L2_mean(h) 	PINN_abs_h_p95 	omega0_rad_s 	omega_est_rad_s 	abs_rel_err_omega(%) 	Notes
0 	NN V1 	97.71 	97.76 		0.0 						data-driven
1 	NN V2 	97.25 	97.14 		0.0 						data-driven
2 	NN V3 	97.71 	97.6 		0.0 						data-driven
3 	PINN V1 					1.205e-03 	6.660e-02 	25.132742 	43.181559 	71.81 	physics residual; SDOF consistency


[INFO] Ecrits dans: /home/a.riyahi/spinn_project/outputs/tables/table1_summary.csv
[INFO] Sources NN — metrics_comparison: ['metrics_comparison.csv', 'metrics_comparison.csv', 'metrics_comparison_rounded.csv']
[INFO] Sources NN — metrics_per_class: ['metrics_per_class.csv', 'metrics_per_class.csv', 'metrics_per_class_rounded.csv']
[INFO] Sources NN — false_negatives  : ['false_negatives.csv']
[INFO] Sources PINN — KPI: pinn_kpis_20251110_192717.json | DIAG: pinn_diag_20251110_192717.npz


SyntaxError: invalid character '—' (U+2014) (2939386952.py, line 12)